# Infosys Springboard Internship 7.0 - Batch 4
## Milestone 1: User Authentication Module

**Tech Stack:** Python, Streamlit, SQLite, bcrypt, PyJWT, Gmail SMTP, Email OTP, ngrok, Google Colab

---

### Overview
This interactive notebook provides the complete, modular implementation for **Milestone 1** of the Infosys Springboard Internship 7.0 (Batch 4). It is designed specifically for the educational evaluation and demonstration environment.

### Core Capabilities:
1. **Signup with Mandatory Email OTP Verification**: Permanent account committed only after successful signup OTP verification.
2. **Dual Identifier Login**: Authenticate with Username or Email and Password.
3. **JWT Session Management**: Issues cryptographically signed HS256 tokens upon login.
4. **Dual-Route Forgot Password**: Route A (Security Question) and Route B (Email OTP).
5. **Role-Based Dashboards**: Protected User Dashboard and Protected Admin Dashboard.
6. **Security Hardening**: Zero hardcoded secrets, bcrypt hashing with salt, rate-limiting and brute-force lockout, credential masking (no password hashes exposed).
7. **Automated Tests**: Comprehensive pytest suite covering auth, OTP, tokens, and validators.

---

### Colab Secrets Configuration:
In Google Colab, click the **Secrets** icon in the left sidebar and configure the following:
- `NGROK_AUTHTOKEN`: Authtoken from https://dashboard.ngrok.com
- `EMAIL_ADDRESS`: Gmail address used to dispatch verification codes
- `EMAIL_PASSWORD`: 16-character Gmail App Password
- `JWT_SECRET`: **Required.** Random secret string used to sign session tokens
- `ADMIN_USERNAME`: (Optional) Admin username (defaults to `admin`)
- `ADMIN_PASSWORD`: **Required.** Admin password (stored hashed in the database)

Ensure 'Notebook access' is enabled for each secret.

In [ ]:
# Step 1: Install required packages in Google Colab
!pip install streamlit pyjwt bcrypt python-dotenv pyngrok pytest pandas -q
print("[OK] Dependencies successfully installed.")

In [ ]:
# Step 2: Verify Colab Secrets Configuration
try:
    from google.colab import userdata
    secrets_needed = ["NGROK_AUTHTOKEN", "EMAIL_ADDRESS", "EMAIL_PASSWORD", "JWT_SECRET", "ADMIN_PASSWORD"]
    missing = []
    for s in secrets_needed:
        try:
            val = userdata.get(s)
            if not val:
                missing.append(s)
            else:
                print(f"[OK] Secret '{s}' is configured.")
        except Exception:
            missing.append(s)

    if missing:
        print(f"\n[WARNING] The following Colab Secrets were not found: {missing}")
        print("Please set them via the Secrets sidebar and toggle 'Notebook access' ON.")
    else:
        print("\n[READY] All required secrets are verified and ready.")
except ImportError:
    print("[INFO] Running outside Google Colab. Using environment variables / .env fallback.")

In [ ]:
# Step 3: Scaffold Modular Directory Structure & Light Mode Configuration
import os
os.makedirs("src", exist_ok=True)
os.makedirs("tests", exist_ok=True)
os.makedirs(".streamlit", exist_ok=True)
with open("src/__init__.py", "w") as f:
    f.write('"""Infosys Milestone 1 Module Package"""\n')
with open("tests/__init__.py", "w") as f:
    f.write('"""Tests Package"""\n')
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase = "light"\nprimaryColor = "#1a6ef5"\nbackgroundColor = "#f6f8fb"\nsecondaryBackgroundColor = "#ffffff"\ntextColor = "#0d1117"\n\n[server]\nheadless = true\nenableCORS = false\nenableXsrfProtection = false\nenableWebsocketCompression = false\n\n[browser]\ngatherUsageStats = false\n')
print("[OK] Directories and light mode theme configuration created.")


### Module: `src/config.py`
Configuration loader with Colab Secrets prioritization.

In [ ]:
%%writefile src/config.py
"""
Configuration and secrets management module for Milestone 1.

Designed for educational evaluation in dual environments:
- Google Colab (retrieves secrets via google.colab.userdata)
- Local Development (retrieves from environment variables or .env file)
"""

import os
from pathlib import Path

# Load local .env if available
try:
    from dotenv import load_dotenv
    load_dotenv()
    load_dotenv(Path(__file__).resolve().parent / ".env")
    load_dotenv(Path(__file__).resolve().parent.parent / ".env")
except ImportError:
    pass

def get_secret(key: str, default: str = "") -> str:
    """
    Retrieve secret prioritizing Google Colab userdata, then environment variables.
    Never exposes plaintext secrets in logs or exceptions.
    Supports common typos (e.g. \\DDRESS or ADDRESS for EMAIL_ADDRESS).
    """
    candidates = [key]
    if key == "EMAIL_ADDRESS":
        candidates.extend(["\\DDRESS", "DDRESS", "ADDRESS", "EMAIL", "GMAIL_ADDRESS", "SENDER_EMAIL"])
    elif key == "EMAIL_PASSWORD":
        candidates.extend(["EMAIL_PASS", "GMAIL_PASSWORD", "APP_PASSWORD", "GMAIL_APP_PASSWORD"])

    # 1. Try Google Colab userdata
    try:
        from google.colab import userdata  # type: ignore
        for k in candidates:
            try:
                val = userdata.get(k)
                if val is not None and str(val).strip():
                    return str(val).strip()
            except Exception:
                pass
    except Exception:
        pass

    # 2. Try OS environment variables
    for k in candidates:
        env_val = os.getenv(k)
        if env_val is not None and env_val.strip():
            return env_val.strip()

    return default

# Base directories
BASE_DIR = Path(__file__).resolve().parent.parent
SRC_DIR = BASE_DIR / "src"

# Database Configuration
DEFAULT_DB_PATH = str(BASE_DIR / "milestone1.db")
DB_PATH = get_secret("DB_PATH", DEFAULT_DB_PATH)

# Authentication & JWT Configuration
JWT_SECRET = get_secret("JWT_SECRET")
JWT_ALGORITHM = "HS256"
JWT_EXPIRATION_HOURS = 2

# Gmail SMTP Configuration
EMAIL_ADDRESS = get_secret("EMAIL_ADDRESS", "").strip()
EMAIL_PASSWORD = get_secret("EMAIL_PASSWORD", "").replace(" ", "").strip()
SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT_SSL = 465
SMTP_PORT_TLS = 587

# OTP Configuration
OTP_EXPIRY_MINUTES = 5
MAX_OTP_ATTEMPTS = 3
OTP_RESEND_COOLDOWN_SECONDS = 60

# Rate Limiting & Lockout
MAX_LOGIN_ATTEMPTS = 3
LOCKOUT_SECONDS = 300  # 5 minutes lockout

# Admin Credentials (read from Colab Secrets / environment only; no password default)
ADMIN_USERNAME = get_secret("ADMIN_USERNAME", "admin")
ADMIN_PASSWORD = get_secret("ADMIN_PASSWORD")
ADMIN_EMAIL = get_secret("ADMIN_EMAIL", "admin@infosys.local")

_missing = [name for name, val in (("JWT_SECRET", JWT_SECRET), ("ADMIN_PASSWORD", ADMIN_PASSWORD)) if not val]
if _missing:
    raise RuntimeError(
        "Missing required secret(s): " + ", ".join(_missing)
        + ". Add them in Colab Secrets (key icon, Notebook access ON) or in a .env file."
    )

# Roles (Milestone 2 builds role-based access control on these)
ROLES = ["Admin", "Franchise Owner", "Store Manager", "Staff", "Auditor"]
DEFAULT_ROLE = "Staff"
ADMIN_ROLE = "Admin"

# ngrok Authtoken
NGROK_AUTHTOKEN = get_secret("NGROK_AUTHTOKEN", "")

# Fixed list of Security Questions for Signup
SECURITY_QUESTIONS = [
    "What was the name of your first school?",
    "What is your mother's maiden name?",
    "What was your favorite childhood pet's name?",
    "What is the name of the city where you were born?",
    "What was the make of your family's first car?"
]


### Module: `src/database.py`
Single unified SQLite database schema (`users`, `pending_signups`, `otp_codes`, `login_attempts`).

In [ ]:
%%writefile src/database.py
"""
Database management module for Milestone 1.

Implements a single unified SQLite database for:
1. Permanent user accounts (`users`)
2. Staged unverified registrations (`pending_signups`)
3. One-Time Passwords (`otp_codes`)
4. Rate limiting and lockout counters (`login_attempts`)
"""

import secrets
import sqlite3
import datetime
from pathlib import Path
from typing import Optional
try:
    from .config import DB_PATH, ADMIN_USERNAME, ADMIN_PASSWORD, ADMIN_EMAIL, ADMIN_ROLE, DEFAULT_ROLE
    from .security import hash_secret
except ImportError:
    try:
        from src.config import DB_PATH, ADMIN_USERNAME, ADMIN_PASSWORD, ADMIN_EMAIL, ADMIN_ROLE, DEFAULT_ROLE
        from src.security import hash_secret
    except ImportError:
        from config import DB_PATH, ADMIN_USERNAME, ADMIN_PASSWORD, ADMIN_EMAIL, ADMIN_ROLE, DEFAULT_ROLE
        from security import hash_secret

def get_connection(db_path: Optional[str] = None) -> sqlite3.Connection:
    """
    Returns an active SQLite connection configured with row factory.
    """
    path = db_path or DB_PATH
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(path, check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn

def init_db(db_path: Optional[str] = None) -> None:
    """
    Initializes all required tables and indexes in the single SQLite database.
    """
    conn = get_connection(db_path)
    cursor = conn.cursor()

    # 1. Permanent Users Table (created only after successful signup OTP verification)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL COLLATE NOCASE,
            email TEXT UNIQUE NOT NULL COLLATE NOCASE,
            password_hash TEXT NOT NULL,
            security_question TEXT NOT NULL,
            security_answer_hash TEXT NOT NULL,
            created_at TEXT NOT NULL,
            updated_at TEXT NOT NULL
        )
    """)
    # Role and outlet columns (added for role-based access control); also upgrades older databases
    existing_cols = {row["name"] for row in cursor.execute("PRAGMA table_info(users)").fetchall()}
    if "role" not in existing_cols:
        cursor.execute(f"ALTER TABLE users ADD COLUMN role TEXT NOT NULL DEFAULT '{DEFAULT_ROLE}'")
    if "outlet" not in existing_cols:
        cursor.execute("ALTER TABLE users ADD COLUMN outlet TEXT")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_users_username ON users(username)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_users_email ON users(email)")

    # 2. Staged Pending Signups Table (temporary storage awaiting signup OTP)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS pending_signups (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL COLLATE NOCASE,
            email TEXT UNIQUE NOT NULL COLLATE NOCASE,
            password_hash TEXT NOT NULL,
            security_question TEXT NOT NULL,
            security_answer_hash TEXT NOT NULL,
            created_at TEXT NOT NULL,
            expires_at REAL NOT NULL
        )
    """)
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_pending_email ON pending_signups(email)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_pending_username ON pending_signups(username)")

    # 3. OTP Codes Table (strictly separates SIGNUP and FORGOT_PASSWORD purposes)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS otp_codes (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            identifier TEXT NOT NULL COLLATE NOCASE,
            otp_hash TEXT NOT NULL,
            purpose TEXT NOT NULL,
            attempts INTEGER DEFAULT 0,
            max_attempts INTEGER DEFAULT 3,
            expires_at REAL NOT NULL,
            is_used INTEGER DEFAULT 0,
            created_at TEXT NOT NULL
        )
    """)
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_otp_lookup ON otp_codes(identifier, purpose, is_used)")

    # 4. Login Attempts Table (rate limiting & brute-force lockout)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS login_attempts (
            identifier TEXT PRIMARY KEY COLLATE NOCASE,
            failed_attempts INTEGER DEFAULT 0,
            last_failed_at REAL DEFAULT 0,
            locked_until REAL DEFAULT 0
        )
    """)

    # 5. Seed the Admin account (password comes from the ADMIN_PASSWORD secret, stored hashed)
    cursor.execute("SELECT id FROM users WHERE username = ? COLLATE NOCASE", (ADMIN_USERNAME,))
    if not cursor.fetchone():
        now_str = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
        cursor.execute("""
            INSERT INTO users (username, email, password_hash, security_question, security_answer_hash,
                               created_at, updated_at, role)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, (ADMIN_USERNAME, ADMIN_EMAIL.lower(), hash_secret(ADMIN_PASSWORD), "N/A",
              hash_secret(secrets.token_urlsafe(24)), now_str, now_str, ADMIN_ROLE))

    conn.commit()
    conn.close()

def clean_expired_pending_signups(current_time: float, db_path: Optional[str] = None) -> None:
    """
    Prunes expired unverified pending signups so users can re-attempt signup
    if their previous session expired.
    """
    conn = get_connection(db_path)
    cursor = conn.cursor()
    cursor.execute("DELETE FROM pending_signups WHERE expires_at < ?", (current_time,))
    conn.commit()
    conn.close()


### Module: `src/security.py`
Salted bcrypt hashing and PyJWT token generation and verification.

In [ ]:
%%writefile src/security.py
"""
Security and cryptography module for Milestone 1.

Implements:
1. Salted bcrypt hashing and verification for passwords and security answers.
2. PyJWT session token generation, decoding, and signature verification.
"""

import bcrypt
import datetime
import jwt
from typing import Tuple, Optional, Dict, Any
try:
    from .config import JWT_SECRET, JWT_ALGORITHM, JWT_EXPIRATION_HOURS
except ImportError:
    try:
        from src.config import JWT_SECRET, JWT_ALGORITHM, JWT_EXPIRATION_HOURS
    except ImportError:
        from config import JWT_SECRET, JWT_ALGORITHM, JWT_EXPIRATION_HOURS

def hash_secret(secret_text: str) -> str:
    """
    Hashes a plaintext string using bcrypt with an automatically generated salt.
    Returns the string representation of the hash.
    """
    if not isinstance(secret_text, str):
        secret_text = str(secret_text)
    salt = bcrypt.gensalt(rounds=12)
    hashed_bytes = bcrypt.hashpw(secret_text.encode("utf-8"), salt)
    return hashed_bytes.decode("utf-8")

def verify_secret(secret_text: str, stored_hash: str) -> bool:
    """
    Verifies a plaintext string against a stored bcrypt hash using constant-time check.
    """
    if not secret_text or not stored_hash:
        return False
    try:
        return bcrypt.checkpw(secret_text.encode("utf-8"), stored_hash.encode("utf-8"))
    except Exception:
        return False

def hash_security_answer(answer: str) -> str:
    """
    Normalizes security answer (trimmed and lowercase) before hashing with bcrypt
    so answering with differing capitalization succeeds securely.
    """
    normalized = answer.strip().lower()
    return hash_secret(normalized)

def verify_security_answer(candidate_answer: str, stored_hash: str) -> bool:
    """
    Verifies normalized candidate security answer against stored hash.
    """
    normalized = candidate_answer.strip().lower()
    return verify_secret(normalized, stored_hash)

def generate_jwt_token(user_id: int, username: str, email: str, expiry_hours: int = JWT_EXPIRATION_HOURS,
                       role: str = "Staff") -> str:
    """
    Generates a cryptographically signed JWT token upon successful user authentication.
    """
    now = datetime.datetime.now(datetime.timezone.utc)
    expiration = now + datetime.timedelta(hours=expiry_hours)

    payload = {
        "sub": str(user_id),
        "username": username,
        "email": email,
        "role": role,
        "iat": int(now.timestamp()),
        "exp": int(expiration.timestamp()),
        "iss": "infosys-springboard-auth"
    }

    token = jwt.encode(payload, JWT_SECRET, algorithm=JWT_ALGORITHM)
    return token

def verify_jwt_token(token: str) -> Tuple[bool, Optional[Dict[str, Any]], str]:
    """
    Decodes and validates a JWT session token.
    Returns: (is_valid, payload, error_message)
    """
    if not token or not isinstance(token, str):
        return False, None, "Session token is missing."

    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=[JWT_ALGORITHM])
        return True, payload, "Valid session."
    except jwt.ExpiredSignatureError:
        return False, None, "Session has expired. Please log in again."
    except jwt.InvalidTokenError as e:
        return False, None, f"Invalid session token: {str(e)}"
    except Exception as e:
        return False, None, f"Token verification error: {str(e)}"


### Module: `src/validators.py`
Field validations, email regex (ab@cd.ef shape rule), and password complexity.

In [ ]:
%%writefile src/validators.py
"""
Validation rules module for Milestone 1.

Implements all field validation requirements specified by Infosys Springboard 7.0:
1. Mandatory fields verification
2. Email format rule: >= 2 chars before @, >= 2 chars between @ and dot, >= 2 chars after final dot
3. Password complexity: >= 8 chars, >= 1 uppercase, >= 1 lowercase, >= 1 digit, >= 1 special symbol
4. Password confirmation matching
"""

import re
from typing import Tuple, Dict, Any

SPECIAL_CHARACTERS = set("!@#$%^&*(),.?\":{}|<>~`_-+=/\\[];'")

def validate_mandatory_fields(fields: Dict[str, Any]) -> Tuple[bool, str]:
    """
    Checks that none of the provided form fields are empty or whitespace-only.
    """
    for field_name, value in fields.items():
        if value is None:
            return False, f"'{field_name}' is required."
        if isinstance(value, str) and not value.strip():
            return False, f"'{field_name}' cannot be empty."
    return True, ""

def validate_email_format(email: str) -> Tuple[bool, str]:
    """
    Validates email format against the official Infosys Milestone 1 rule:
    - At least 2 characters before the '@'
    - At least 2 characters between the '@' and the dot
    - At least 2 characters after the final dot
    Example compliant shape: ab@cd.ef
    """
    if not email or not isinstance(email, str):
        return False, "Email address is required."

    email = email.strip()
    if "@" not in email:
        return False, "Email must contain an '@' symbol."

    parts = email.split("@")
    if len(parts) != 2:
        return False, "Email must contain exactly one '@' symbol."

    local_part, domain_part = parts[0], parts[1]

    # Rule 1: At least 2 characters before the @
    if len(local_part) < 2:
        return False, "Email must have at least 2 characters before the '@' symbol."

    if "." not in domain_part:
        return False, "Email domain must contain at least one '.' (dot)."

    domain_subparts = domain_part.split(".")
    # Intermediate domain name (between @ and first dot) must be at least 2 characters
    if len(domain_subparts[0]) < 2:
        return False, "Email domain must have at least 2 characters between '@' and the dot."

    # Final domain extension (after final dot) must be at least 2 characters
    final_extension = domain_subparts[-1]
    if len(final_extension) < 2:
        return False, "Email domain must have at least 2 characters after the final dot."

    # Standard general syntax check
    email_regex = r"^[a-zA-Z0-9_.+-]{2,}@[a-zA-Z0-9-]+(?:\.[a-zA-Z0-9-]+)*\.[a-zA-Z]{2,}$"
    if not re.match(email_regex, email):
        return False, "Invalid email format. Format must match pattern like ab@cd.ef"

    return True, ""

def validate_password_complexity(password: str) -> Tuple[bool, str]:
    """
    Validates password against Infosys complexity requirements:
    - Minimum 8 characters
    - At least one uppercase letter
    - At least one lowercase letter
    - At least one number
    - At least one special symbol
    """
    if not password or not isinstance(password, str):
        return False, "Password is required."

    if len(password) < 8:
        return False, "Password must be at least 8 characters long."

    has_upper = any(c.isupper() for c in password)
    if not has_upper:
        return False, "Password must contain at least one uppercase letter (A-Z)."

    has_lower = any(c.islower() for c in password)
    if not has_lower:
        return False, "Password must contain at least one lowercase letter (a-z)."

    has_digit = any(c.isdigit() for c in password)
    if not has_digit:
        return False, "Password must contain at least one numeric digit (0-9)."

    has_special = any(c in SPECIAL_CHARACTERS for c in password)
    if not has_special:
        return False, "Password must contain at least one special symbol (e.g. !@#$%^&*)."

    return True, ""

def validate_password_confirmation(password: str, confirm_password: str) -> Tuple[bool, str]:
    """
    Validates that password and confirm password match exactly.
    """
    if password != confirm_password:
        return False, "Confirm Password does not match Password."
    return True, ""

def validate_username(username: str) -> Tuple[bool, str]:
    """
    Validates username format (min 3 chars, alphanumeric and underscores).
    """
    if not username or not isinstance(username, str):
        return False, "Username is required."
    username = username.strip()
    if len(username) < 3:
        return False, "Username must be at least 3 characters long."
    if not re.match(r"^[a-zA-Z0-9_]+$", username):
        return False, "Username may only contain letters, numbers, and underscores."
    return True, ""


### Module: `src/otp_service.py`
6-digit secure OTP generation, 5-minute expiry, 3-attempt limits, and Gmail SMTP integration.

In [ ]:
%%writefile src/otp_service.py
"""
OTP and Email notification service module for Milestone 1.

Features:
1. Cryptographically secure 6-digit OTP generation via secrets module.
2. Distinct OTP contexts ('SIGNUP' vs 'FORGOT_PASSWORD').
3. Database-backed tracking with bcrypt hash storage, 5-minute expiry, and 3-attempt limit.
4. Gmail SMTP dispatch with anti-spam RFC headers and dual text/HTML templates.
5. Simulated dispatch mode for offline automated testing and test suites.
"""

import time
import secrets
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.utils import formatdate, make_msgid
from typing import Tuple, Optional

try:
    from .config import (
        EMAIL_ADDRESS,
        EMAIL_PASSWORD,
        SMTP_SERVER,
        SMTP_PORT_SSL,
        SMTP_PORT_TLS,
        OTP_EXPIRY_MINUTES,
        MAX_OTP_ATTEMPTS,
        OTP_RESEND_COOLDOWN_SECONDS,
        DB_PATH
    )
    from .database import get_connection
    from .security import hash_secret, verify_secret
except ImportError:
    try:
        from src.config import (
            EMAIL_ADDRESS,
            EMAIL_PASSWORD,
            SMTP_SERVER,
            SMTP_PORT_SSL,
            SMTP_PORT_TLS,
            OTP_EXPIRY_MINUTES,
            MAX_OTP_ATTEMPTS,
            OTP_RESEND_COOLDOWN_SECONDS,
            DB_PATH
        )
        from src.database import get_connection
        from src.security import hash_secret, verify_secret
    except ImportError:
        from config import (
            EMAIL_ADDRESS,
            EMAIL_PASSWORD,
            SMTP_SERVER,
            SMTP_PORT_SSL,
            SMTP_PORT_TLS,
            OTP_EXPIRY_MINUTES,
            MAX_OTP_ATTEMPTS,
            OTP_RESEND_COOLDOWN_SECONDS,
            DB_PATH
        )
        from database import get_connection
        from security import hash_secret, verify_secret

OTP_LOCKOUT_SECONDS = 15 * 60  # 15 minutes lockout for 3 OTP mismatches

def generate_numeric_otp(digits: int = 6) -> str:
    """
    Generates a cryptographically secure numeric OTP code.
    """
    lower = 10 ** (digits - 1)
    upper = 10 ** digits - 1
    return str(secrets.randbelow(upper - lower + 1) + lower)

def create_and_store_otp(
    identifier: str,
    purpose: str,
    db_path: Optional[str] = None
) -> Tuple[bool, str, str]:
    """
    Creates an OTP for a specified identifier (email) and purpose ('SIGNUP' or 'FORGOT_PASSWORD').
    Stores hashed OTP in database with attempt tracking and expiration.

    Returns: (success, plaintext_otp_or_empty, status_message)
    """
    norm_id = identifier.strip().lower()
    purpose = purpose.strip().upper()
    now_ts = time.time()
    conn = get_connection(db_path)
    cursor = conn.cursor()

    # Check if this identifier is currently blocked/locked in login_attempts
    cursor.execute("SELECT locked_until FROM login_attempts WHERE identifier = ?", (norm_id,))
    lock_row = cursor.fetchone()
    if lock_row and lock_row["locked_until"] and lock_row["locked_until"] > now_ts:
        remaining_mins = max(1, int((lock_row["locked_until"] - now_ts + 59) // 60))
        conn.close()
        return False, "", f"Account is temporarily blocked due to 3 failed OTP attempts. No OTP requests or operations allowed for {remaining_mins} more minute(s)."

    # Check resend cooldown from the most recent active OTP for this identifier & purpose
    cursor.execute("""
        SELECT created_at, expires_at, is_used
        FROM otp_codes
        WHERE identifier = ? AND purpose = ? AND is_used = 0
        ORDER BY id DESC LIMIT 1
    """, (norm_id, purpose))
    recent = cursor.fetchone()

    if recent:
        try:
            pass
        except Exception:
            pass

    # Invalidate any previous unused OTPs for this identifier and purpose
    cursor.execute("""
        UPDATE otp_codes
        SET is_used = 1
        WHERE identifier = ? AND purpose = ? AND is_used = 0
    """, (norm_id, purpose))

    # Generate new 6-digit OTP
    otp_code = generate_numeric_otp(6)
    hashed_otp = hash_secret(otp_code)
    expires_at = now_ts + (OTP_EXPIRY_MINUTES * 60)
    created_at = time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime(now_ts))

    cursor.execute("""
        INSERT INTO otp_codes (identifier, otp_hash, purpose, attempts, max_attempts, expires_at, is_used, created_at)
        VALUES (?, ?, ?, 0, ?, ?, 0, ?)
    """, (norm_id, hashed_otp, purpose, MAX_OTP_ATTEMPTS, expires_at, created_at))

    conn.commit()
    conn.close()

    return True, otp_code, "OTP generated successfully."

def verify_stored_otp(
    identifier: str,
    purpose: str,
    candidate_otp: str,
    db_path: Optional[str] = None
) -> Tuple[bool, str]:
    """
    Verifies candidate OTP for the identifier and purpose.
    Enforces expiry, max attempt limits, single-use consumption,
    and locks the account for 15 minutes after 3 failed attempts.

    Returns: (is_valid, message)
    """
    norm_id = identifier.strip().lower()
    purpose = purpose.strip().upper()
    candidate_otp = candidate_otp.strip()
    now_ts = time.time()

    if not candidate_otp:
        return False, "OTP code is required."

    conn = get_connection(db_path)
    cursor = conn.cursor()

    # Check if account is already locked
    cursor.execute("SELECT locked_until FROM login_attempts WHERE identifier = ?", (norm_id,))
    lock_row = cursor.fetchone()
    if lock_row and lock_row["locked_until"] and lock_row["locked_until"] > now_ts:
        remaining_mins = max(1, int((lock_row["locked_until"] - now_ts + 59) // 60))
        conn.close()
        return False, f"Account is temporarily blocked due to 3 failed OTP attempts. No operations allowed for {remaining_mins} more minute(s)."

    cursor.execute("""
        SELECT id, otp_hash, attempts, max_attempts, expires_at, is_used
        FROM otp_codes
        WHERE identifier = ? AND purpose = ? AND is_used = 0
        ORDER BY id DESC LIMIT 1
    """, (norm_id, purpose))
    record = cursor.fetchone()

    if not record:
        conn.close()
        return False, "No active OTP request found. Please request a new OTP code."

    record_id = record["id"]
    stored_hash = record["otp_hash"]
    attempts = record["attempts"]
    max_attempts = record["max_attempts"]
    expires_at = record["expires_at"]

    # 1. Check expiration
    if now_ts > expires_at:
        cursor.execute("UPDATE otp_codes SET is_used = 1 WHERE id = ?", (record_id,))
        conn.commit()
        conn.close()
        return False, f"OTP expired after {OTP_EXPIRY_MINUTES} minutes. Please request a new OTP code."

    # 2. Check attempt limits
    if attempts >= max_attempts:
        cursor.execute("UPDATE otp_codes SET is_used = 1 WHERE id = ?", (record_id,))
        conn.commit()
        conn.close()
        return False, f"Maximum verification attempts ({max_attempts}) exceeded. This OTP is invalidated. Please request a new code."

    # 3. Verify OTP hash
    if verify_secret(candidate_otp, stored_hash):
        # Mark as consumed (one-time verification)
        cursor.execute("UPDATE otp_codes SET is_used = 1 WHERE id = ?", (record_id,))
        conn.commit()
        conn.close()
        return True, "OTP verified successfully."
    else:
        new_attempts = attempts + 1
        remaining = max_attempts - new_attempts
        if remaining <= 0:
            # 3 mismatches: Invalidate OTP and block account for 15 minutes!
            locked_until = now_ts + OTP_LOCKOUT_SECONDS
            cursor.execute("UPDATE otp_codes SET attempts = ?, is_used = 1 WHERE id = ?", (new_attempts, record_id))
            cursor.execute("""
                INSERT INTO login_attempts (identifier, failed_attempts, last_failed_at, locked_until)
                VALUES (?, ?, ?, ?)
                ON CONFLICT(identifier) DO UPDATE SET
                    failed_attempts = excluded.failed_attempts,
                    last_failed_at = excluded.last_failed_at,
                    locked_until = excluded.locked_until
            """, (norm_id, new_attempts, now_ts, locked_until))

            # Also lock associated username and email from users table
            cursor.execute("SELECT username, email FROM users WHERE email = ? COLLATE NOCASE OR username = ? COLLATE NOCASE", (norm_id, norm_id))
            u_rows = cursor.fetchall()
            for u_row in u_rows:
                for ident in [u_row["username"].lower(), u_row["email"].lower()]:
                    cursor.execute("""
                        INSERT INTO login_attempts (identifier, failed_attempts, last_failed_at, locked_until)
                        VALUES (?, ?, ?, ?)
                        ON CONFLICT(identifier) DO UPDATE SET
                            failed_attempts = excluded.failed_attempts,
                            last_failed_at = excluded.last_failed_at,
                            locked_until = excluded.locked_until
                    """, (ident, new_attempts, now_ts, locked_until))

            conn.commit()
            conn.close()
            return False, f"Incorrect OTP code. Maximum attempts ({max_attempts}) exceeded. Account is blocked for 15 minutes. No further OTP requests or operations allowed."
        else:
            cursor.execute("UPDATE otp_codes SET attempts = ? WHERE id = ?", (new_attempts, record_id))
            conn.commit()
            conn.close()
            return False, f"Incorrect OTP code. {remaining} attempt(s) remaining."

def send_otp_email(to_email: str, otp_code: str, purpose: str) -> Tuple[bool, str]:
    """
    Sends the OTP code via Gmail SMTP with RFC headers and dual text/HTML templates.
    If Gmail credentials are not configured, provides a descriptive message for local/demo runs.
    """
    try:
        from src.config import get_secret
    except ImportError:
        try:
            from .config import get_secret
        except ImportError:
            from config import get_secret

    sender = get_secret("EMAIL_ADDRESS", EMAIL_ADDRESS).strip().lower()
    raw_pass = get_secret("EMAIL_PASSWORD", EMAIL_PASSWORD)
    password = raw_pass.replace(" ", "").replace("-", "").strip()

    if not sender or not password:
        return False, (
            "Gmail credentials (EMAIL_ADDRESS / EMAIL_PASSWORD) not configured in Colab Secrets or .env. "
            f"[Demo Mock Mode: Your OTP code is {otp_code}]"
        )

    action_title = "Signup Verification" if purpose.upper() == "SIGNUP" else "Password Recovery"
    subject = f"Infosys Portal - {action_title} OTP Code: {otp_code}"

    # Build anti-spam multipart alternative email
    msg = MIMEMultipart("alternative")
    msg["From"] = f"Infosys Portal <{sender}>"
    msg["To"] = to_email
    msg["Subject"] = subject
    msg["Date"] = formatdate(localtime=True)
    msg["Message-ID"] = make_msgid()
    msg["Reply-To"] = sender

    # Plaintext fallback
    plain_text = (
        f"Infosys Portal - {action_title}\n\n"
        f"Your one-time verification code is: {otp_code}\n\n"
        f"This code will expire in {OTP_EXPIRY_MINUTES} minutes.\n"
        f"You have a maximum of {MAX_OTP_ATTEMPTS} verification attempts.\n"
        f"If you did not request this code, please ignore this email.\n"
    )

    # Clean enterprise HTML template
    html_content = f"""<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8">
  <style>
    body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; background-color: #f8fafc; margin: 0; padding: 30px; }}
    .card {{ max-width: 520px; margin: 0 auto; background: #ffffff; border-radius: 8px; border: 1px solid #e2e8f0; padding: 32px; box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.05); }}
    .header {{ border-bottom: 1px solid #f1f5f9; padding-bottom: 16px; margin-bottom: 24px; }}
    .brand {{ font-size: 11px; font-weight: 700; letter-spacing: 0.1em; color: #007cc3; text-transform: uppercase; margin-bottom: 4px; }}
    .title {{ font-size: 20px; font-weight: 600; color: #0f172a; margin: 0; }}
    .badge {{ display: inline-block; background-color: #f0f9ff; color: #0369a1; padding: 3px 8px; border-radius: 4px; font-weight: 600; font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; border: 1px solid #bae6fd; margin-top: 8px; }}
    .otp-box {{ background-color: #f8fafc; border: 1px solid #cbd5e1; border-radius: 6px; text-align: center; padding: 22px; margin: 24px 0; }}
    .otp-code {{ font-size: 32px; font-weight: 700; letter-spacing: 6px; color: #0f172a; font-family: Consolas, Monaco, monospace; }}
    .footer {{ font-size: 12px; color: #64748b; text-align: center; margin-top: 28px; border-top: 1px solid #f1f5f9; padding-top: 16px; line-height: 1.5; }}
  </style>
</head>
<body>
  <div class="card">
    <div class="header">
      <div class="brand">Infosys Springboard</div>
      <div class="title">Verification Code</div>
      <div class="badge">{action_title}</div>
    </div>
    <p style="color: #334155; font-size: 14px; line-height: 1.5;">Hello,</p>
    <p style="color: #334155; font-size: 14px; line-height: 1.5;">
      Use the following verification code to proceed with your {action_title.lower()} request:
    </p>
    <div class="otp-box">
      <div class="otp-code">{otp_code}</div>
    </div>
    <p style="font-size: 13px; color: #64748b; line-height: 1.5;">
      This code expires in <strong>{OTP_EXPIRY_MINUTES} minutes</strong> and allows up to <strong>{MAX_OTP_ATTEMPTS} verification attempts</strong>.
    </p>
    <p style="font-size: 12.5px; color: #94a3b8; line-height: 1.5;">
      If you did not initiate this request, no further action is required.
    </p>
    <div class="footer">
      Infosys Springboard &bull; Milestone 1 Verification Service
    </div>
  </div>
</body>
</html>"""

    msg.attach(MIMEText(plain_text, "plain", "utf-8"))
    msg.attach(MIMEText(html_content, "html", "utf-8"))

    try:
        # First attempt via SSL Port 465
        with smtplib.SMTP_SSL(SMTP_SERVER, SMTP_PORT_SSL, timeout=12) as server:
            server.login(sender, password)
            server.sendmail(sender, [to_email], msg.as_string())
        return True, "Verification email sent successfully."
    except Exception as ssl_err:
        try:
            # Fallback attempt via TLS Port 587
            with smtplib.SMTP(SMTP_SERVER, SMTP_PORT_TLS, timeout=12) as server:
                server.ehlo()
                server.starttls()
                server.ehlo()
                server.login(sender, password)
                server.sendmail(sender, [to_email], msg.as_string())
            return True, "Verification email sent successfully."
        except Exception as tls_err:
            return False, f"Failed to send email via SMTP: {str(ssl_err)} / {str(tls_err)}"


### Module: `src/auth_service.py`
Authentication workflows: OTP-gated signup, username/email login, brute-force lockout, dual-route forgot password, admin operations.

In [ ]:
%%writefile src/auth_service.py
"""
Authentication and user management service for Milestone 1.

Responsibilities:
1. Mandatory Email OTP gated signup:
   - Stages unverified user in `pending_signups`
   - Dispatches 'SIGNUP' OTP
   - Commits permanent account to `users` only upon valid OTP entry
2. Login with dual identifier (Username or Email):
   - Rate limiting and brute-force lockout
   - Constant-time bcrypt password matching
   - Generic error messages (no enumeration)
   - Cryptographic JWT session token generation
3. Forgot Password recovery:
   - Route A: Security Question verification
   - Route B: Email OTP verification ('FORGOT_PASSWORD' context)
4. Admin authentication & masked registered user directory:
   - Strictly hides password and security answer hashes
"""

import time
from typing import Tuple, Optional, Dict, Any, List

try:
    from .config import (
        ADMIN_USERNAME,
        ADMIN_PASSWORD,
        MAX_LOGIN_ATTEMPTS,
        LOCKOUT_SECONDS,
        SECURITY_QUESTIONS,
        DB_PATH
    )
    from .database import get_connection, clean_expired_pending_signups
    from .security import (
        hash_secret,
        verify_secret,
        hash_security_answer,
        verify_security_answer,
        generate_jwt_token
    )
    from .validators import (
        validate_mandatory_fields,
        validate_username,
        validate_email_format,
        validate_password_complexity,
        validate_password_confirmation
    )
    from .otp_service import (
        create_and_store_otp,
        verify_stored_otp,
        send_otp_email
    )
except ImportError:
    try:
        from src.config import (
            ADMIN_USERNAME,
            ADMIN_PASSWORD,
            MAX_LOGIN_ATTEMPTS,
            LOCKOUT_SECONDS,
            SECURITY_QUESTIONS,
            DB_PATH
        )
        from src.database import get_connection, clean_expired_pending_signups
        from src.security import (
            hash_secret,
            verify_secret,
            hash_security_answer,
            verify_security_answer,
            generate_jwt_token
        )
        from src.validators import (
            validate_mandatory_fields,
            validate_username,
            validate_email_format,
            validate_password_complexity,
            validate_password_confirmation
        )
        from src.otp_service import (
            create_and_store_otp,
            verify_stored_otp,
            send_otp_email
        )
    except ImportError:
        from config import (
            ADMIN_USERNAME,
            ADMIN_PASSWORD,
            MAX_LOGIN_ATTEMPTS,
            LOCKOUT_SECONDS,
            SECURITY_QUESTIONS,
            DB_PATH
        )
        from database import get_connection, clean_expired_pending_signups
        from security import (
            hash_secret,
            verify_secret,
            hash_security_answer,
            verify_security_answer,
            generate_jwt_token
        )
        from validators import (
            validate_mandatory_fields,
            validate_username,
            validate_email_format,
            validate_password_complexity,
            validate_password_confirmation
        )
        from otp_service import (
            create_and_store_otp,
            verify_stored_otp,
            send_otp_email
        )

def register_user(
    username: str,
    email: str,
    password: str,
    confirm_password: str,
    security_question: str,
    security_answer: str,
    db_path: Optional[str] = None
) -> Tuple[bool, str]:
    """
    Registers a permanent user account directly into the SQLite database.
    No email OTP verification is required.
    Returns: (success, message)
    """
    # 1. Validate mandatory fields
    fields = {
        "Username": username,
        "Email": email,
        "Password": password,
        "Confirm Password": confirm_password,
        "Security Question": security_question,
        "Security Answer": security_answer
    }
    is_valid, msg = validate_mandatory_fields(fields)
    if not is_valid:
        return False, msg

    # 2. Validate formats
    is_valid, msg = validate_username(username)
    if not is_valid:
        return False, msg

    is_valid, msg = validate_email_format(email)
    if not is_valid:
        return False, msg

    is_valid, msg = validate_password_complexity(password)
    if not is_valid:
        return False, msg

    is_valid, msg = validate_password_confirmation(password, confirm_password)
    if not is_valid:
        return False, msg

    if security_question not in SECURITY_QUESTIONS:
        return False, "Please select a valid security question from the provided list."

    norm_username = username.strip()
    norm_email = email.strip().lower()

    conn = get_connection(db_path)
    cursor = conn.cursor()

    # 3. Check permanent users table for duplicate username or email
    cursor.execute("SELECT id FROM users WHERE username = ? COLLATE NOCASE", (norm_username,))
    if cursor.fetchone():
        conn.close()
        return False, "Username is already registered. Please choose a different username."

    cursor.execute("SELECT id FROM users WHERE email = ? COLLATE NOCASE", (norm_email,))
    if cursor.fetchone():
        conn.close()
        return False, "Email address is already registered. Please use another email or log in."

    # 4. Hash credentials
    password_hash = hash_secret(password)
    answer_hash = hash_security_answer(security_answer)

    # 5. Insert directly into users
    now_ts = time.time()
    created_at = time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime(now_ts))
    try:
        cursor.execute("""
            INSERT INTO users (username, email, password_hash, security_question, security_answer_hash, created_at, updated_at)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (norm_username, norm_email, password_hash, security_question, answer_hash, created_at, created_at))
        conn.commit()
        conn.close()
        return True, "Account successfully registered! You can now sign in with your credentials."
    except Exception as e:
        conn.close()
        return False, f"Failed to create account: {str(e)}"

def stage_signup(
    username: str,
    email: str,
    password: str,
    confirm_password: str,
    security_question: str,
    security_answer: str,
    db_path: Optional[str] = None
) -> Tuple[bool, str, Optional[str]]:
    """
    Validates signup details, ensures username/email are unique in permanent accounts,
    stages the unverified account in `pending_signups`, and dispatches an OTP.

    Permanent account is NOT created here.
    Returns: (success, message, dev_otp_for_offline_runs)
    """
    # 1. Validate mandatory fields
    fields = {
        "Username": username,
        "Email": email,
        "Password": password,
        "Confirm Password": confirm_password,
        "Security Question": security_question,
        "Security Answer": security_answer
    }
    is_valid, msg = validate_mandatory_fields(fields)
    if not is_valid:
        return False, msg, None

    # 2. Validate formats
    is_valid, msg = validate_username(username)
    if not is_valid:
        return False, msg, None

    is_valid, msg = validate_email_format(email)
    if not is_valid:
        return False, msg, None

    is_valid, msg = validate_password_complexity(password)
    if not is_valid:
        return False, msg, None

    is_valid, msg = validate_password_confirmation(password, confirm_password)
    if not is_valid:
        return False, msg, None

    if security_question not in SECURITY_QUESTIONS:
        return False, "Please select a valid security question from the provided list.", None

    norm_username = username.strip()
    norm_email = email.strip().lower()

    conn = get_connection(db_path)
    cursor = conn.cursor()

    # 3. Check permanent users table for duplicate username or email
    cursor.execute("SELECT id FROM users WHERE username = ? COLLATE NOCASE", (norm_username,))
    if cursor.fetchone():
        conn.close()
        return False, "Username is already registered. Please choose a different username.", None

    cursor.execute("SELECT id FROM users WHERE email = ? COLLATE NOCASE", (norm_email,))
    if cursor.fetchone():
        conn.close()
        return False, "Email address is already registered. Please use another email or log in.", None

    # 4. Clean expired pending signups
    now_ts = time.time()
    clean_expired_pending_signups(now_ts, db_path)

    # Invalidate any previous pending signup for this email/username
    cursor.execute("DELETE FROM pending_signups WHERE email = ? OR username = ?", (norm_email, norm_username))

    # 5. Hash credentials
    password_hash = hash_secret(password)
    answer_hash = hash_security_answer(security_answer)

    # 6. Stage in pending_signups (valid for 15 minutes)
    expires_at = now_ts + (15 * 60)
    created_at = time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime(now_ts))

    cursor.execute("""
        INSERT INTO pending_signups (username, email, password_hash, security_question, security_answer_hash, created_at, expires_at)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (norm_username, norm_email, password_hash, security_question, answer_hash, created_at, expires_at))
    conn.commit()
    conn.close()

    # 7. Generate and send SIGNUP OTP
    otp_ok, otp_code, otp_msg = create_and_store_otp(norm_email, "SIGNUP", db_path)
    if not otp_ok:
        return False, f"Failed to generate signup OTP: {otp_msg}", None

    email_ok, email_msg = send_otp_email(norm_email, otp_code, "SIGNUP")

    status_note = f"Verification OTP sent to {norm_email}."
    if not email_ok:
        status_note += f" [Notice: {email_msg}]"

    return True, status_note, otp_code

def complete_signup(
    email: str,
    otp_code: str,
    db_path: Optional[str] = None
) -> Tuple[bool, str]:
    """
    Verifies the signup OTP code. Upon successful verification, moves the staged
    registration into the permanent `users` table and clears the pending record.
    """
    norm_email = email.strip().lower()
    otp_code = otp_code.strip()

    if not norm_email:
        return False, "Email address is required."
    if not otp_code:
        return False, "Please enter the OTP verification code."

    # 1. Verify OTP with 'SIGNUP' purpose
    is_valid, msg = verify_stored_otp(norm_email, "SIGNUP", otp_code, db_path)
    if not is_valid:
        return False, msg

    conn = get_connection(db_path)
    cursor = conn.cursor()

    # 2. Retrieve staged registration
    now_ts = time.time()
    cursor.execute("""
        SELECT username, email, password_hash, security_question, security_answer_hash, expires_at
        FROM pending_signups
        WHERE email = ?
    """, (norm_email,))
    staged = cursor.fetchone()

    if not staged:
        conn.close()
        return False, "No pending registration found for this email, or it may have already been completed."

    if now_ts > staged["expires_at"]:
        cursor.execute("DELETE FROM pending_signups WHERE email = ?", (norm_email,))
        conn.commit()
        conn.close()
        return False, "Pending registration has expired. Please fill out the signup form again."

    # 3. Commit to permanent users table
    created_at = time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime(now_ts))
    try:
        cursor.execute("""
            INSERT INTO users (username, email, password_hash, security_question, security_answer_hash, created_at, updated_at)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (
            staged["username"],
            staged["email"],
            staged["password_hash"],
            staged["security_question"],
            staged["security_answer_hash"],
            created_at,
            created_at
        ))
        # Remove staged entry
        cursor.execute("DELETE FROM pending_signups WHERE email = ?", (norm_email,))
        conn.commit()
        conn.close()
        return True, "Account successfully verified and created! You can now log in."
    except Exception as e:
        conn.close()
        return False, f"Failed to finalize account creation: {str(e)}"

def login_user(
    identifier: str,
    password: str,
    db_path: Optional[str] = None
) -> Tuple[bool, Optional[str], str, Optional[Dict[str, Any]]]:
    """
    Authenticates user using Username OR Email and password.
    Enforces rate-limiting and brute-force lockout.
    Returns generic error messages on authentication failure to prevent enumeration.

    Returns: (success, jwt_token_or_none, message, user_info_dict)
    """
    norm_id = identifier.strip()
    if not norm_id or not password:
        return False, None, "Username/Email and password are both mandatory.", None

    conn = get_connection(db_path)
    cursor = conn.cursor()
    now_ts = time.time()

    # 1. Check rate-limit & lockout status
    cursor.execute("SELECT failed_attempts, locked_until FROM login_attempts WHERE identifier = ?", (norm_id.lower(),))
    attempt_row = cursor.fetchone()

    if attempt_row and attempt_row["locked_until"] and attempt_row["locked_until"] > now_ts:
        remaining_secs = max(1, int(attempt_row["locked_until"] - now_ts))
        remaining_mins = max(1, int((attempt_row["locked_until"] - now_ts + 59) // 60))
        conn.close()
        return False, None, f"Account is temporarily locked due to security violations. Please wait {remaining_secs} seconds ({remaining_mins} minute(s)) before trying again.", None

    # 2. Look up user by username OR email
    cursor.execute("""
        SELECT id, username, email, password_hash, role, outlet, created_at
        FROM users
        WHERE username = ? COLLATE NOCASE OR email = ? COLLATE NOCASE
    """, (norm_id, norm_id))
    user_row = cursor.fetchone()

    if not user_row:
        # Increment failed attempts for this identifier
        _record_failed_login(cursor, norm_id.lower(), attempt_row, now_ts)
        conn.commit()
        conn.close()
        return False, None, "Invalid username/email or password.", None

    # 3. Verify password hash
    stored_hash = user_row["password_hash"]
    if not verify_secret(password, stored_hash):
        _record_failed_login(cursor, norm_id.lower(), attempt_row, now_ts)
        conn.commit()
        conn.close()
        return False, None, "Invalid username/email or password.", None

    # 4. Successful Login: clear login attempts
    cursor.execute("DELETE FROM login_attempts WHERE identifier = ?", (norm_id.lower(),))
    cursor.execute("DELETE FROM login_attempts WHERE identifier = ?", (user_row["username"].lower(),))
    cursor.execute("DELETE FROM login_attempts WHERE identifier = ?", (user_row["email"].lower(),))
    conn.commit()
    conn.close()

    # 5. Issue JWT session token
    user_id = user_row["id"]
    username = user_row["username"]
    email = user_row["email"]
    role = user_row["role"]
    token = generate_jwt_token(user_id=user_id, username=username, email=email, role=role)

    user_info = {
        "id": user_id,
        "username": username,
        "email": email,
        "role": role,
        "outlet": user_row["outlet"],
        "created_at": user_row["created_at"]
    }

    return True, token, "Login successful.", user_info

def _record_failed_login(cursor, identifier: str, attempt_row, now_ts: float):
    current_attempts = (attempt_row["failed_attempts"] if attempt_row else 0) + 1
    locked_until = 0.0

    if current_attempts >= MAX_LOGIN_ATTEMPTS:
        locked_until = now_ts + LOCKOUT_SECONDS

    cursor.execute("""
        INSERT INTO login_attempts (identifier, failed_attempts, last_failed_at, locked_until)
        VALUES (?, ?, ?, ?)
        ON CONFLICT(identifier) DO UPDATE SET
            failed_attempts = excluded.failed_attempts,
            last_failed_at = excluded.last_failed_at,
            locked_until = excluded.locked_until
    """, (identifier, current_attempts, now_ts, locked_until))

# --- Forgot Password: Route A (Security Question) ---

def get_security_question_for_user(
    identifier: str,
    db_path: Optional[str] = None
) -> Tuple[bool, Optional[str], str]:
    """
    Retrieves the security question stored during signup for the given username OR email.
    Also checks if the account is temporarily locked out.
    """
    norm_id = identifier.strip()
    if not norm_id:
        return False, None, "Username or Email address is required."

    conn = get_connection(db_path)
    cursor = conn.cursor()
    now_ts = time.time()

    # Check if account is locked
    cursor.execute("SELECT locked_until FROM login_attempts WHERE identifier = ?", (norm_id.lower(),))
    lock_row = cursor.fetchone()
    if lock_row and lock_row["locked_until"] and lock_row["locked_until"] > now_ts:
        remaining_mins = max(1, int((lock_row["locked_until"] - now_ts + 59) // 60))
        conn.close()
        return False, None, f"Account is temporarily locked. No operations allowed for {remaining_mins} more minute(s)."

    cursor.execute("""
        SELECT security_question, username, email
        FROM users
        WHERE username = ? COLLATE NOCASE OR email = ? COLLATE NOCASE
    """, (norm_id, norm_id))
    row = cursor.fetchone()
    conn.close()

    if not row:
        return False, None, "User not found with that username or email address."

    return True, row["security_question"], "Security question retrieved."

def reset_password_with_security_question(
    identifier: str,
    security_answer: str,
    new_password: str,
    confirm_password: str,
    db_path: Optional[str] = None
) -> Tuple[bool, str]:
    """
    Verifies user's security answer (by username OR email) and updates password if complex and confirmed.
    Also checks if the account is temporarily locked out.
    """
    norm_id = identifier.strip()
    norm_answer = security_answer.strip()

    fields = {
        "Username or Email": norm_id,
        "Security Answer": norm_answer,
        "New Password": new_password,
        "Confirm Password": confirm_password
    }
    is_valid, msg = validate_mandatory_fields(fields)
    if not is_valid:
        return False, msg

    is_valid, msg = validate_password_complexity(new_password)
    if not is_valid:
        return False, msg

    is_valid, msg = validate_password_confirmation(new_password, confirm_password)
    if not is_valid:
        return False, msg

    conn = get_connection(db_path)
    cursor = conn.cursor()
    now_ts = time.time()

    # Check if account is locked
    cursor.execute("SELECT locked_until FROM login_attempts WHERE identifier = ?", (norm_id.lower(),))
    lock_row = cursor.fetchone()
    if lock_row and lock_row["locked_until"] and lock_row["locked_until"] > now_ts:
        remaining_mins = max(1, int((lock_row["locked_until"] - now_ts + 59) // 60))
        conn.close()
        return False, f"Account is temporarily locked. No operations allowed for {remaining_mins} more minute(s)."

    cursor.execute("""
        SELECT id, username, email, security_answer_hash
        FROM users
        WHERE username = ? COLLATE NOCASE OR email = ? COLLATE NOCASE
    """, (norm_id, norm_id))
    user_row = cursor.fetchone()

    if not user_row:
        conn.close()
        return False, "User not found."

    # Also check if user's actual username or email is locked
    cursor.execute("SELECT locked_until FROM login_attempts WHERE identifier = ? OR identifier = ?", (user_row["username"].lower(), user_row["email"].lower()))
    user_lock = cursor.fetchone()
    if user_lock and user_lock["locked_until"] and user_lock["locked_until"] > now_ts:
        remaining_mins = max(1, int((user_lock["locked_until"] - now_ts + 59) // 60))
        conn.close()
        return False, f"Account is temporarily locked. No operations allowed for {remaining_mins} more minute(s)."

    stored_answer_hash = user_row["security_answer_hash"]
    if not verify_security_answer(norm_answer, stored_answer_hash):
        conn.close()
        return False, "Incorrect security answer."

    new_hash = hash_secret(new_password)
    now_str = time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime(time.time()))

    cursor.execute("""
        UPDATE users
        SET password_hash = ?, updated_at = ?
        WHERE id = ?
    """, (new_hash, now_str, user_row["id"]))
    conn.commit()
    conn.close()

    return True, "Password has been successfully reset! You can now log in with your new password."

# --- Forgot Password: Route B (Email OTP) ---

def initiate_forgot_password_otp(
    email: str,
    db_path: Optional[str] = None
) -> Tuple[bool, str, Optional[str]]:
    """
    Initiates password recovery via Email OTP for a registered email.
    """
    norm_email = email.strip().lower()
    is_valid, msg = validate_email_format(norm_email)
    if not is_valid:
        return False, msg, None

    conn = get_connection(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT id FROM users WHERE email = ? COLLATE NOCASE", (norm_email,))
    user_row = cursor.fetchone()
    conn.close()

    if not user_row:
        return False, "No registered account found with this email address.", None

    otp_ok, otp_code, otp_msg = create_and_store_otp(norm_email, "FORGOT_PASSWORD", db_path)
    if not otp_ok:
        return False, otp_msg, None

    email_ok, email_msg = send_otp_email(norm_email, otp_code, "FORGOT_PASSWORD")

    status_note = f"Password recovery OTP sent to {norm_email}."
    if not email_ok:
        status_note += f" [Notice: {email_msg}]"

    return True, status_note, otp_code

def reset_password_with_otp(
    email: str,
    otp_code: str,
    new_password: str,
    confirm_password: str,
    db_path: Optional[str] = None
) -> Tuple[bool, str]:
    """
    Verifies the recovery OTP and updates the user's password.
    """
    norm_email = email.strip().lower()

    fields = {
        "Email": norm_email,
        "OTP Code": otp_code,
        "New Password": new_password,
        "Confirm Password": confirm_password
    }
    is_valid, msg = validate_mandatory_fields(fields)
    if not is_valid:
        return False, msg

    is_valid, msg = validate_password_complexity(new_password)
    if not is_valid:
        return False, msg

    is_valid, msg = validate_password_confirmation(new_password, confirm_password)
    if not is_valid:
        return False, msg

    # Verify 'FORGOT_PASSWORD' OTP
    is_valid, msg = verify_stored_otp(norm_email, "FORGOT_PASSWORD", otp_code, db_path)
    if not is_valid:
        return False, msg

    conn = get_connection(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT id FROM users WHERE email = ? COLLATE NOCASE", (norm_email,))
    user_row = cursor.fetchone()

    if not user_row:
        conn.close()
        return False, "User account not found."

    new_hash = hash_secret(new_password)
    now_str = time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime(time.time()))

    cursor.execute("""
        UPDATE users
        SET password_hash = ?, updated_at = ?
        WHERE id = ?
    """, (new_hash, now_str, user_row["id"]))
    conn.commit()
    conn.close()

    return True, "Password has been successfully reset! You can now log in with your new password."

# --- Admin Operations ---

def authenticate_admin(username: str, password: str) -> bool:
    """
    Authenticates administrative user against secure configuration.
    """
    if not username or not password:
        return False
    return (username.strip() == ADMIN_USERNAME) and (password.strip() == ADMIN_PASSWORD)

def get_all_registered_users(db_path: Optional[str] = None) -> List[Dict[str, Any]]:
    """
    Returns registered users catalog for Admin Dashboard.
    Strictly excludes password hashes and security answer hashes.
    """
    conn = get_connection(db_path)
    cursor = conn.cursor()
    cursor.execute("""
        SELECT id, username, email, role, outlet, created_at, updated_at
        FROM users
        ORDER BY id ASC
    """)
    rows = cursor.fetchall()
    conn.close()

    users_list = []
    for r in rows:
        users_list.append({
            "User ID": r["id"],
            "Username": r["username"],
            "Email Address": r["email"],
            "Role": r["role"],
            "Outlet": r["outlet"] or "-",
            "Registration Date": r["created_at"],
            "Status": "Active"
        })
    return users_list


### Module: `src/app.py`
Streamlit UI application featuring bespoke light mode, Josefin Sans typography, and protected dashboards.

In [ ]:
%%writefile src/app.py
"""
Infosys Springboard Internship 7.0 - Batch 4
Milestone 1: User Authentication Module (Streamlit Application)

Professional light-mode design with Josefin Sans typography.
Clean, modern, single-card corporate authentication portal.

Features:
- Pure light-mode aesthetic, no dark mode
- Josefin Sans typography and crisp card UI
- Unified sign-in for both standard users and administrators
- Single centered login card with form and header merged
- Standard login page pattern: bottom links for "Sign Up" and "Forgot Password"
- Direct account registration without email OTP
- Account locked for 15 minutes upon 3 OTP mismatches
- Password recovery via Security Question (lookup by username or email) or Email OTP
- Protected User and Administrator dashboards with JWT verification
- No sidebar panel: clean centered authentication layout
"""

import sys
import os
import time
from pathlib import Path

CURRENT_DIR = Path(__file__).resolve().parent
if str(CURRENT_DIR) not in sys.path:
    sys.path.insert(0, str(CURRENT_DIR))
if str(CURRENT_DIR.parent) not in sys.path:
    sys.path.insert(0, str(CURRENT_DIR.parent))

import streamlit as st
import pandas as pd

try:
    from .config import SECURITY_QUESTIONS, EMAIL_ADDRESS, ADMIN_USERNAME
    from .database import init_db
    from .security import verify_jwt_token
    from .auth_service import (
        register_user,
        stage_signup,
        complete_signup,
        login_user,
        get_security_question_for_user,
        reset_password_with_security_question,
        initiate_forgot_password_otp,
        reset_password_with_otp,
        authenticate_admin,
        get_all_registered_users
    )
except ImportError:
    try:
        from src.config import SECURITY_QUESTIONS, EMAIL_ADDRESS, ADMIN_USERNAME
        from src.database import init_db
        from src.security import verify_jwt_token
        from src.auth_service import (
            register_user,
            stage_signup,
            complete_signup,
            login_user,
            get_security_question_for_user,
            reset_password_with_security_question,
            initiate_forgot_password_otp,
            reset_password_with_otp,
            authenticate_admin,
            get_all_registered_users
        )
    except ImportError:
        from config import SECURITY_QUESTIONS, EMAIL_ADDRESS, ADMIN_USERNAME
        from database import init_db
        from security import verify_jwt_token
        from auth_service import (
            register_user,
            stage_signup,
            complete_signup,
            login_user,
            get_security_question_for_user,
            reset_password_with_security_question,
            initiate_forgot_password_otp,
            reset_password_with_otp,
            authenticate_admin,
            get_all_registered_users
        )

# Ensure database is initialized
init_db()

st.set_page_config(
    page_title="Infosys Springboard — Identity Platform",
    layout="wide",
    initial_sidebar_state="collapsed"
)

GLOBAL_CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Josefin+Sans:wght@300;400;600;700&family=Inter:wght@400;500;600&display=swap');

/* ─── Reset & Base ─────────────────────────────────── */
html, body, .stApp {
    background-color: #f8fafc !important;
    font-family: 'Inter', system-ui, -apple-system, sans-serif !important;
    color: #0f172a !important;
}

header[data-testid="stHeader"] {
    background: transparent !important;
}

.stDeployButton,
header[data-testid="stHeader"] [data-testid="stDecoration"] {
    display: none !important;
}

/* Hide sidebar panel completely */
section[data-testid="stSidebar"],
[data-testid="collapsedControl"] {
    display: none !important;
}

.block-container {
    padding-top: 3.2rem !important;
    padding-bottom: 4rem !important;
    max-width: 1040px !important;
    margin: 0 auto !important;
}

/* ─── Typography ────────────────────────────────────── */
h1, h2, h3, h4, h5, h6 {
    font-family: 'Josefin Sans', sans-serif !important;
    color: #0f172a !important;
    letter-spacing: -0.01em !important;
}

/* ─── Form Inputs ───────────────────────────────────── */
div[data-baseweb="input"] {
    background-color: #ffffff !important;
    border: 1.5px solid #cbd5e1 !important;
    border-radius: 8px !important;
    transition: border-color 0.15s, box-shadow 0.15s !important;
    height: 44px !important;
}

div[data-baseweb="input"]:focus-within {
    border-color: #1a6ef5 !important;
    box-shadow: 0 0 0 3px rgba(26, 110, 245, 0.12) !important;
}

input, textarea {
    color: #0f172a !important;
    -webkit-text-fill-color: #0f172a !important;
    font-family: 'Inter', sans-serif !important;
    font-size: 14px !important;
}

input::placeholder {
    color: #94a3b8 !important;
    -webkit-text-fill-color: #94a3b8 !important;
}

div[data-baseweb="select"] > div {
    background-color: #ffffff !important;
    border: 1.5px solid #cbd5e1 !important;
    border-radius: 8px !important;
    color: #0f172a !important;
    min-height: 44px !important;
}

label[data-testid="stWidgetLabel"] p,
label[data-testid="stWidgetLabel"] span {
    font-family: 'Josefin Sans', sans-serif !important;
    font-size: 12px !important;
    font-weight: 600 !important;
    color: #475569 !important;
    letter-spacing: 0.05em !important;
    text-transform: uppercase !important;
    margin-bottom: 4px !important;
}

/* ─── Forms Styled as Single Bordered Container ──────── */
div[data-testid="stForm"] {
    background: #ffffff !important;
    border: 1px solid #e2e8f0 !important;
    border-radius: 14px !important;
    padding: 40px 36px 32px 36px !important;
    box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.05), 0 2px 4px -2px rgba(0, 0, 0, 0.05) !important;
}

div[data-testid="stForm"] div[data-testid="stVerticalBlock"] { gap: 1rem !important; }
div[data-testid="stFormSubmitButton"],
div[data-testid="stElementContainer"]:has(div[data-testid="stFormSubmitButton"]),
div[data-testid="stFormSubmitButton"] > div { width: 100% !important; margin-top: 6px; }
div[data-testid="stColumn"] div[data-testid="stElementContainer"] { width: 100% !important; }
div[data-testid="stColumn"]:last-child .stButton { display: flex; justify-content: flex-end; }
div[data-testid="stFormSubmitButton"] > button { width: 100% !important; }

/* ─── Buttons ────────────────────────────────────────── */
button[kind="primary"],
div[data-testid="stFormSubmitButton"] button {
    display: inline-flex !important;
    align-items: center !important;
    justify-content: center !important;
    background-color: #0f172a !important;
    color: #ffffff !important;
    border: none !important;
    border-radius: 8px !important;
    font-family: 'Josefin Sans', sans-serif !important;
    font-size: 13.5px !important;
    font-weight: 700 !important;
    letter-spacing: 0.08em !important;
    text-transform: uppercase !important;
    height: 44px !important;
    min-height: 44px !important;
    padding: 0 24px !important;
    width: 100% !important;
    cursor: pointer !important;
    box-shadow: 0 1px 3px rgba(0,0,0,0.08) !important;
    transition: all 0.15s ease !important;
    margin-top: 8px !important;
}

button[kind="secondary"],
.stButton > button {
    display: inline-flex !important;
    align-items: center !important;
    justify-content: center !important;
    background-color: #ffffff !important;
    color: #475569 !important;
    border: 1px solid #cbd5e1 !important;
    border-radius: 8px !important;
    font-family: 'Josefin Sans', sans-serif !important;
    font-size: 12.5px !important;
    font-weight: 600 !important;
    letter-spacing: 0.04em !important;
    height: 38px !important;
    min-height: 38px !important;
    padding: 0 16px !important;
    width: 100% !important;
    cursor: pointer !important;
    transition: all 0.15s ease !important;
}

div.stButton > button[kind="tertiary"] {
    background: transparent !important;
    border: none !important;
    box-shadow: none !important;
    color: #1a6ef5 !important;
    font-family: 'Inter', sans-serif !important;
    font-size: 13.5px !important;
    font-weight: 500 !important;
    letter-spacing: 0 !important;
    width: auto !important;
    padding: 0 !important;
}
div.stButton > button[kind="tertiary"]:hover {
    background: transparent !important;
    text-decoration: underline !important;
    color: #0f4fc4 !important;
}

button[kind="primary"]:hover,
div[data-testid="stFormSubmitButton"] button:hover {
    background-color: #1a6ef5 !important;
    box-shadow: 0 4px 12px rgba(26, 110, 245, 0.25) !important;
    transform: translateY(-1px) !important;
    color: #ffffff !important;
}

button[kind="secondary"]:hover,
.stButton > button:hover {
    background-color: #f8fafc !important;
    border-color: #1a6ef5 !important;
    color: #1a6ef5 !important;
}

/* ─── Tabs ──────────────────────────────────────────── */
div[data-baseweb="tab-list"] {
    background: #f1f5f9 !important;
    border-radius: 10px !important;
    padding: 4px !important;
    border: 1px solid #e2e8f0 !important;
    gap: 4px !important;
    margin-bottom: 16px !important;
}

button[data-baseweb="tab"] {
    font-family: 'Josefin Sans', sans-serif !important;
    font-size: 12.5px !important;
    font-weight: 600 !important;
    color: #64748b !important;
    letter-spacing: 0.04em !important;
    text-transform: uppercase !important;
    border-radius: 7px !important;
    border: none !important;
    background: transparent !important;
    padding: 8px 16px !important;
    transition: all 0.15s ease !important;
}

button[data-baseweb="tab"][aria-selected="true"] {
    background: #ffffff !important;
    color: #1a6ef5 !important;
    box-shadow: 0 1px 3px rgba(0,0,0,0.08) !important;
}

/* ─── Metrics ───────────────────────────────────────── */
div[data-testid="stMetric"] {
    background: #ffffff !important;
    border: 1px solid #e2e8f0 !important;
    border-radius: 10px !important;
    padding: 18px 22px !important;
}

div[data-testid="stMetricLabel"] p {
    font-family: 'Josefin Sans', sans-serif !important;
    font-size: 11px !important;
    font-weight: 700 !important;
    color: #64748b !important;
    letter-spacing: 0.10em !important;
    text-transform: uppercase !important;
}

div[data-testid="stMetricValue"] {
    font-family: 'Josefin Sans', sans-serif !important;
    font-size: 26px !important;
    font-weight: 700 !important;
    color: #0f172a !important;
}

/* ─── DataTable ─────────────────────────────────────── */
div[data-testid="stDataFrame"] {
    border: 1px solid #e2e8f0 !important;
    border-radius: 10px !important;
    overflow: hidden !important;
}

/* ─── Card component (for Dashboards) ────────────────── */
.ui-card {
    background: #ffffff;
    border: 1px solid #e2e8f0;
    border-radius: 14px;
    padding: 28px 28px;
    margin-bottom: 20px;
    box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.05);
}

/* ─── Label/Badge ───────────────────────────────────── */
.ui-label {
    display: inline-block;
    font-family: 'Josefin Sans', sans-serif;
    font-size: 10.5px;
    font-weight: 700;
    letter-spacing: 0.14em;
    text-transform: uppercase;
    padding: 4px 10px;
    border-radius: 5px;
    margin-bottom: 12px;
}

.ui-label-green {
    background: #ecfdf5;
    color: #059669;
    border: 1px solid #a7f3d0;
}

.ui-label-orange {
    background: #fff7ed;
    color: #c2410c;
    border: 1px solid #fed7aa;
}

/* ─── Info box ──────────────────────────────────────── */
.ui-infobox {
    background: #eff6ff;
    border: 1px solid #bfdbfe;
    border-left: 4px solid #1a6ef5;
    border-radius: 6px;
    padding: 10px 14px;
    font-size: 12.5px;
    color: #1e40af;
    line-height: 1.5;
    margin: 4px 0 14px 0;
}
</style>
"""

st.markdown(GLOBAL_CSS, unsafe_allow_html=True)

# ─── Session State Initialization ─────────────────────────────────────────────
if "auth_view" not in st.session_state:
    st.session_state["auth_view"] = "sign_in"
if "jwt_token" not in st.session_state:
    st.session_state["jwt_token"] = None
if "user_info" not in st.session_state:
    st.session_state["user_info"] = None
if "admin_logged_in" not in st.session_state:
    st.session_state["admin_logged_in"] = False
if "signup_success_msg" not in st.session_state:
    st.session_state["signup_success_msg"] = None

def set_auth_width(px):
    """Narrow, centered column so the auth card reads like a real login card."""
    st.markdown(
        f"<style>.block-container{{max-width:{px}px !important;padding-left:1rem !important;padding-right:1rem !important;}}</style>",
        unsafe_allow_html=True,
    )

def logout_user():
    st.session_state["jwt_token"] = None
    st.session_state["user_info"] = None
    st.session_state["auth_view"] = "sign_in"
    st.rerun()

def logout_admin():
    st.session_state["admin_logged_in"] = False
    st.session_state["user_info"] = None
    st.session_state["auth_view"] = "sign_in"
    st.rerun()

# Active session checks
is_user_authenticated = False
if st.session_state["jwt_token"]:
    is_valid, payload, _ = verify_jwt_token(st.session_state["jwt_token"])
    if is_valid and payload:
        is_user_authenticated = True
    else:
        st.session_state["jwt_token"] = None
        st.session_state["user_info"] = None

is_admin_authenticated = st.session_state["admin_logged_in"]

# ═════════════════════════════════════════════════════════════════════════════
# VIEW: PROTECTED USER DASHBOARD
# ═════════════════════════════════════════════════════════════════════════════
if is_user_authenticated:
    user = st.session_state["user_info"] or {}

    st.markdown("""
    <div class="ui-card">
        <span class="ui-label ui-label-green">Authenticated Session &bull; JWT Verified</span>
        <h1 style="font-size: 26px; margin: 8px 0 6px 0;">Welcome, {username}</h1>
        <p style="color: #64748b; font-size: 14px; margin: 0;">
            Your session is secured with a cryptographically signed JSON Web Token (HS256).
        </p>
    </div>
    """.format(username=user.get("username", "User")), unsafe_allow_html=True)

    col_profile, col_session = st.columns([3, 2])

    with col_profile:
        st.markdown("""
        <div class="ui-card">
            <div style="font-family: 'Josefin Sans', sans-serif; font-size: 10.5px; font-weight: 700;
                        letter-spacing: 0.12em; color: #64748b; text-transform: uppercase; margin-bottom: 14px;">
                User Profile
            </div>
            <table style="width: 100%; border-collapse: collapse; font-size: 13.5px;">
                <tr style="border-bottom: 1px solid #f1f5f9;">
                    <td style="padding: 12px 0; color: #64748b; width: 38%; font-weight: 500;">Username</td>
                    <td style="padding: 12px 0; font-weight: 600; color: #0f172a;">{username}</td>
                </tr>
                <tr style="border-bottom: 1px solid #f1f5f9;">
                    <td style="padding: 12px 0; color: #64748b; font-weight: 500;">Email</td>
                    <td style="padding: 12px 0; font-weight: 600; color: #0f172a;">{email}</td>
                </tr>
                <tr style="border-bottom: 1px solid #f1f5f9;">
                    <td style="padding: 12px 0; color: #64748b; font-weight: 500;">System ID</td>
                    <td style="padding: 12px 0; font-family: monospace; font-size: 13px; color: #1a6ef5;">#{uid}</td>
                </tr>
                <tr>
                    <td style="padding: 12px 0; color: #64748b; font-weight: 500;">Registered</td>
                    <td style="padding: 12px 0; color: #334155;">{created}</td>
                </tr>
            </table>
            <div style="margin-top: 18px;" class="ui-infobox">
                <strong>Security Status:</strong> Passwords and security answers are stored as
                salted bcrypt hashes — never in plaintext.
            </div>
        </div>
        """.format(
            username=user.get("username", "N/A"),
            email=user.get("email", "N/A"),
            uid=user.get("id", "N/A"),
            created=user.get("created_at", "N/A")
        ), unsafe_allow_html=True)

    with col_session:
        st.markdown("""
        <div class="ui-card">
            <div style="font-family: 'Josefin Sans', sans-serif; font-size: 10.5px; font-weight: 700;
                        letter-spacing: 0.12em; color: #64748b; text-transform: uppercase; margin-bottom: 14px;">
                Session Control
            </div>
            <p style="font-size: 13px; color: #64748b; line-height: 1.6; margin-bottom: 20px;">
                JWT token claims and signature are verified on every page load.
                You can terminate your session securely at any time.
            </p>
        </div>
        """, unsafe_allow_html=True)
        if st.button("Sign Out", key="user_logout_btn"):
            logout_user()

# ═════════════════════════════════════════════════════════════════════════════
# VIEW: PROTECTED ADMINISTRATOR CONSOLE
# ═════════════════════════════════════════════════════════════════════════════
elif is_admin_authenticated:
    st.markdown("""
    <div class="ui-card">
        <span class="ui-label ui-label-orange">Administrative Access &bull; Privileged Console</span>
        <h1 style="font-size: 26px; margin: 8px 0 6px 0;">Administrator Directory</h1>
        <p style="color: #64748b; font-size: 14px; margin: 0;">
            Complete catalog of registered candidate accounts. Password hashes and security answers are strictly masked.
        </p>
    </div>
    """, unsafe_allow_html=True)

    users_data = get_all_registered_users()
    c1, c2, c3 = st.columns(3)
    with c1:
        st.metric("Registered Users", len(users_data))
    with c2:
        st.metric("Database Engine", "SQLite")
    with c3:
        st.metric("Active Admin", ADMIN_USERNAME)

    st.markdown("<div style='height: 12px;'></div>", unsafe_allow_html=True)
    st.markdown("""
    <div style="font-family: 'Josefin Sans', sans-serif; font-size: 10.5px; font-weight: 700;
                letter-spacing: 0.12em; color: #64748b; text-transform: uppercase; margin-bottom: 10px;">
        Registered User Accounts
    </div>
    """, unsafe_allow_html=True)

    if users_data:
        df = pd.DataFrame(users_data)
        st.dataframe(df, use_container_width=True, hide_index=True)
    else:
        st.info("No accounts registered yet in milestone1.db.")

    st.markdown("<div style='height: 16px;'></div>", unsafe_allow_html=True)
    col_btn, _, _ = st.columns([1, 2, 2])
    with col_btn:
        if st.button("Exit Console", key="admin_exit_btn"):
            logout_admin()

# ═════════════════════════════════════════════════════════════════════════════
# VIEW: PUBLIC AUTHENTICATION PORTAL (CLEAN SINGLE-CARD PATTERN)
# ═════════════════════════════════════════════════════════════════════════════
else:
    cur_view = st.session_state["auth_view"]

    # ─────────────────────────────────────────────────────────────────────────
    # SUB-VIEW: SIGN IN (NARROW CENTERED CARD, HEADING & FORM MERGED)
    # ─────────────────────────────────────────────────────────────────────────
    if cur_view == "sign_in":
        set_auth_width(460)
        with st.container():
            with st.form("unified_login_form"):
                st.markdown("""
                <div style="text-align: center; margin-bottom: 22px;">
                    <div style="font-family: 'Josefin Sans', sans-serif; font-size: 11px; font-weight: 700;
                                letter-spacing: 0.18em; color: #1a6ef5; text-transform: uppercase; margin-bottom: 4px;">
                        Infosys Springboard
                    </div>
                    <h2 style="font-size: 24px; font-weight: 700; margin: 0 0 4px 0; color: #0f172a;">Sign In</h2>
                    <p style="color: #64748b; font-size: 13.5px; margin: 0;">Enter your credentials to continue</p>
                </div>
                """, unsafe_allow_html=True)

                identifier = st.text_input("Username or Email Address", placeholder="Enter username or email", key="uni_id")
                password = st.text_input("Password", type="password", placeholder="Enter your password", key="uni_pw")
                login_submit = st.form_submit_button("Sign In")

                if login_submit:
                    # Check for Administrator credentials first
                    if identifier.strip() == ADMIN_USERNAME and authenticate_admin(identifier, password):
                        st.session_state["admin_logged_in"] = True
                        st.session_state["user_info"] = {"username": ADMIN_USERNAME, "role": "Administrator"}
                        st.success("Administrator authenticated.")
                        st.rerun()
                    else:
                        # Standard User login with rate limiting & lockout
                        success, token, msg, user_info = login_user(identifier, password)
                        if success and token:
                            st.session_state["jwt_token"] = token
                            st.session_state["user_info"] = user_info
                            st.success(msg)
                            st.rerun()
                        else:
                            st.error(msg)

            # Standard bottom links
            st.markdown("<div style='height: 14px;'></div>", unsafe_allow_html=True)
            col_b1, col_b2 = st.columns(2)
            with col_b1:
                if st.button("Create an account", key="goto_signup_btn", type="tertiary"):
                    st.session_state["auth_view"] = "create_account"
                    st.rerun()
            with col_b2:
                if st.button("Forgot password?", key="goto_reset_btn", type="tertiary"):
                    st.session_state["auth_view"] = "reset_password"
                    st.rerun()

    # ─────────────────────────────────────────────────────────────────────────
    # SUB-VIEW: CREATE ACCOUNT (SPACIOUS SINGLE CARD — NO EMAIL OTP)
    # ─────────────────────────────────────────────────────────────────────────
    elif cur_view == "create_account":
        set_auth_width(640)
        with st.container():
            with st.form("direct_signup_form"):
                st.markdown("""
                <div style="text-align: center; margin-bottom: 22px;">
                    <div style="font-family: 'Josefin Sans', sans-serif; font-size: 11px; font-weight: 700;
                                letter-spacing: 0.18em; color: #1a6ef5; text-transform: uppercase; margin-bottom: 4px;">
                        Infosys Springboard
                    </div>
                    <h2 style="font-size: 24px; font-weight: 700; margin: 0 0 4px 0; color: #0f172a;">Create Account</h2>
                    <p style="color: #64748b; font-size: 13.5px; margin: 0;">Instant registration with salted bcrypt encryption</p>
                </div>
                """, unsafe_allow_html=True)

                if st.session_state.get("signup_success_msg"):
                    st.success(st.session_state["signup_success_msg"])

                col1, col2 = st.columns(2)
                with col1:
                    username  = st.text_input("Username", placeholder="e.g. alex_dev", key="reg_u")
                    email     = st.text_input("Email Address", placeholder="e.g. alex@company.com", key="reg_e")
                with col2:
                    password  = st.text_input("Password", type="password", placeholder="8+ characters", key="reg_p")
                    confirm_p = st.text_input("Confirm Password", type="password", placeholder="Re-enter password", key="reg_cp")

                security_question = st.selectbox("Security Question", SECURITY_QUESTIONS, key="reg_sq")
                security_answer   = st.text_input("Security Answer", placeholder="Answer used for account recovery", key="reg_sa")

                st.markdown("""
                <div class="ui-infobox" style="margin-top: 8px; margin-bottom: 8px;">
                    <strong>Security Guarantee:</strong> Passwords and security answers are stored exclusively as salted bcrypt hashes.
                </div>
                """, unsafe_allow_html=True)

                signup_submit = st.form_submit_button("Create Account")

                if signup_submit:
                    ok, msg = register_user(
                        username=username,
                        email=email,
                        password=password,
                        confirm_password=confirm_p,
                        security_question=security_question,
                        security_answer=security_answer
                    )
                    if ok:
                        st.session_state["signup_success_msg"] = msg
                        st.rerun()
                    else:
                        st.error(msg)

            st.markdown("<div style='height: 14px;'></div>", unsafe_allow_html=True)
            if st.button("Already have an account? Sign in", key="reg_back_btn", type="tertiary"):
                st.session_state["signup_success_msg"] = None
                st.session_state["auth_view"] = "sign_in"
                st.rerun()

    # ─────────────────────────────────────────────────────────────────────────
    # SUB-VIEW: RESET PASSWORD (SPACIOUS SINGLE CARD WITH TABS)
    # ─────────────────────────────────────────────────────────────────────────
    elif cur_view == "reset_password":
        set_auth_width(560)
        with st.container():
            st.markdown("""
            <div style="text-align: center; margin-bottom: 20px;">
                <div style="font-family: 'Josefin Sans', sans-serif; font-size: 11px; font-weight: 700;
                            letter-spacing: 0.18em; color: #1a6ef5; text-transform: uppercase; margin-bottom: 4px;">
                    Infosys Springboard
                </div>
                <h2 style="font-size: 24px; font-weight: 700; margin: 0 0 4px 0; color: #0f172a;">Password Recovery</h2>
                <p style="color: #64748b; font-size: 13.5px; margin: 0;">Recover access using your security question or email verification code</p>
            </div>
            """, unsafe_allow_html=True)

            if st.session_state.get("recovery_done_msg"):
                st.success(st.session_state.pop("recovery_done_msg"))

            tab_sq, tab_otp = st.tabs(["Security Question", "Email Code"])

            with tab_sq:
                if not st.session_state.get("active_sec_q_user"):
                    with st.form("sec_q_fetch_form"):
                        sq_identifier = st.text_input("Username or Email Address", placeholder="Enter registered username or email", key="sq_uid")
                        fetch_btn = st.form_submit_button("Continue")
                        if fetch_btn:
                            ok, question, msg = get_security_question_for_user(sq_identifier)
                            if ok:
                                st.session_state["active_sec_q_user"] = sq_identifier
                                st.session_state["active_sec_question"] = question
                                st.rerun()
                            else:
                                st.error(msg)
                else:
                    with st.form("sec_q_reset_form"):
                        st.markdown(f"""
                        <div class="ui-infobox">
                            <strong>Security Question:</strong> {st.session_state['active_sec_question']}
                        </div>
                        """, unsafe_allow_html=True)
                        sq_answer    = st.text_input("Security Answer", type="password", placeholder="Enter your answer", key="sq_ans")
                        sq_new_pass  = st.text_input("New Password", type="password", placeholder="Minimum 8 characters", key="sq_np")
                        sq_conf_pass = st.text_input("Confirm New Password", type="password", placeholder="Re-enter new password", key="sq_cp")
                        sq_reset_btn = st.form_submit_button("Update Password")

                        if sq_reset_btn:
                            ok, msg = reset_password_with_security_question(
                                st.session_state["active_sec_q_user"],
                                sq_answer, sq_new_pass, sq_conf_pass
                            )
                            if ok:
                                st.session_state["recovery_done_msg"] = msg
                                del st.session_state["active_sec_question"]
                                del st.session_state["active_sec_q_user"]
                                st.rerun()
                            else:
                                st.error(msg)
                    if st.button("Use a different account", key="sq_change_btn", type="tertiary"):
                        del st.session_state["active_sec_question"]
                        del st.session_state["active_sec_q_user"]
                        st.rerun()

            with tab_otp:
                if not st.session_state.get("active_rec_email"):
                    with st.form("otp_recovery_send_form"):
                        rec_email = st.text_input("Registered Email Address", placeholder="Enter your registered email address", key="rec_em")
                        send_otp_btn = st.form_submit_button("Send Code")

                        if send_otp_btn:
                            ok, msg, dev_otp = initiate_forgot_password_otp(rec_email)
                            if ok:
                                st.session_state["active_rec_email"] = rec_email.strip().lower()
                                st.session_state["rec_send_msg"] = msg
                                st.rerun()
                            else:
                                st.error(msg)
                else:
                    with st.form("otp_recovery_finalize_form"):
                        if st.session_state.get("rec_send_msg"):
                            st.success(st.session_state["rec_send_msg"])
                        st.markdown("""
                        <div class="ui-infobox">
                            3 incorrect codes will block this account for 15 minutes.
                        </div>
                        """, unsafe_allow_html=True)
                        rec_otp      = st.text_input("6-Digit Verification Code", max_chars=6, placeholder="Enter 6-digit code", key="rec_code")
                        rec_new_pass = st.text_input("New Password", type="password", placeholder="Minimum 8 characters", key="rec_np")
                        rec_conf     = st.text_input("Confirm New Password", type="password", placeholder="Re-enter new password", key="rec_cp")
                        rec_reset_btn = st.form_submit_button("Update Password")

                        if rec_reset_btn:
                            ok, msg = reset_password_with_otp(
                                st.session_state["active_rec_email"],
                                rec_otp, rec_new_pass, rec_conf
                            )
                            if ok:
                                st.session_state["recovery_done_msg"] = msg
                                del st.session_state["active_rec_email"]
                                st.rerun()
                            else:
                                st.error(msg)
                    if st.button("Use a different email", key="rec_change_btn", type="tertiary"):
                        del st.session_state["active_rec_email"]
                        st.rerun()

            st.markdown("<div style='height: 14px;'></div>", unsafe_allow_html=True)
            if st.button("Back to sign in", key="reset_back_btn", type="tertiary"):
                st.session_state["auth_view"] = "sign_in"
                st.rerun()


In [ ]:
%%writefile app.py
"""
Infosys Springboard - Milestone 1 Runner
Executes src/app.py directly using runpy so that Streamlit reruns re-execute the UI logic.
"""
import runpy
from pathlib import Path

target = Path(__file__).resolve().parent / "src" / "app.py"
runpy.run_path(str(target), run_name="__main__")


### Automated Test Suite
Writes all unit and integration test files into `tests/`.

In [ ]:
%%writefile tests/test_validators.py
"""
Unit tests for input validation rules according to Infosys Milestone 1 requirements.
"""

import pytest
from src.validators import (
    validate_mandatory_fields,
    validate_email_format,
    validate_password_complexity,
    validate_password_confirmation,
    validate_username
)

def test_mandatory_fields_empty():
    fields = {"Username": "john", "Email": "", "Password": "SecretPassword1@"}
    valid, msg = validate_mandatory_fields(fields)
    assert not valid
    assert "Email" in msg

def test_mandatory_fields_whitespace():
    fields = {"Username": "  ", "Email": "john@example.com"}
    valid, msg = validate_mandatory_fields(fields)
    assert not valid
    assert "Username" in msg

def test_mandatory_fields_success():
    fields = {"Username": "john_doe", "Email": "john@example.com", "Password": "SecretPassword1@"}
    valid, msg = validate_mandatory_fields(fields)
    assert valid
    assert msg == ""

def test_email_format_infosys_rule():
    # Valid shape (at least 2 chars before @, between @ and dot, and after dot)
    valid_emails = [
        "ab@cd.ef",
        "john.doe@company.org",
        "intern2026@infosys.co.in",
        "user_123@domain.ac.uk"
    ]
    for email in valid_emails:
        valid, msg = validate_email_format(email)
        assert valid, f"Expected {email} to be valid, got: {msg}"

    # Invalid: less than 2 chars before @
    valid, msg = validate_email_format("a@cd.ef")
    assert not valid
    assert "at least 2 characters before" in msg

    # Invalid: less than 2 chars between @ and dot
    valid, msg = validate_email_format("ab@c.ef")
    assert not valid
    assert "at least 2 characters between" in msg

    # Invalid: less than 2 chars after final dot
    valid, msg = validate_email_format("ab@cd.e")
    assert not valid
    assert "at least 2 characters after" in msg

    # Invalid: missing @ or dot
    valid, _ = validate_email_format("plainaddress")
    assert not valid

    valid, _ = validate_email_format("user@nodot")
    assert not valid

def test_password_complexity():
    # Valid passwords: >=8 chars, uppercase, lowercase, digit, special symbol
    assert validate_password_complexity("Strong@123")[0]
    assert validate_password_complexity("Infosys#2026")[0]
    assert validate_password_complexity("P@ssw0rd!xyz")[0]

    # Too short (< 8 chars)
    valid, msg = validate_password_complexity("Ab1@")
    assert not valid
    assert "at least 8 characters" in msg

    # Missing uppercase
    valid, msg = validate_password_complexity("strong@123")
    assert not valid
    assert "uppercase" in msg

    # Missing lowercase
    valid, msg = validate_password_complexity("STRONG@123")
    assert not valid
    assert "lowercase" in msg

    # Missing digit
    valid, msg = validate_password_complexity("Strong@Password")
    assert not valid
    assert "numeric digit" in msg

    # Missing special character
    valid, msg = validate_password_complexity("StrongPass123")
    assert not valid
    assert "special symbol" in msg

def test_password_confirmation():
    valid, _ = validate_password_confirmation("Secret123!", "Secret123!")
    assert valid

    valid, msg = validate_password_confirmation("Secret123!", "Different123!")
    assert not valid
    assert "does not match" in msg

def test_username_format():
    assert validate_username("john_doe")[0]
    assert validate_username("user123")[0]
    assert validate_username("abc")[0]

    # Too short (<3 chars)
    valid, _ = validate_username("ab")
    assert not valid

    # Invalid characters (spaces, special symbols)
    valid, _ = validate_username("john doe")
    assert not valid

    valid, _ = validate_username("john@doe")
    assert not valid


In [ ]:
%%writefile tests/test_security.py
"""
Unit tests for security, bcrypt hashing, and JWT tokens.
"""

import time
import jwt
from src.security import (
    hash_secret,
    verify_secret,
    hash_security_answer,
    verify_security_answer,
    generate_jwt_token,
    verify_jwt_token
)
from src.config import JWT_SECRET, JWT_ALGORITHM

def test_bcrypt_hashing_and_verification():
    raw_password = "SecretPassword@123"
    hashed = hash_secret(raw_password)

    # Hash should be non-empty and start with $2b$
    assert hashed.startswith("$2b$")
    assert hashed != raw_password

    # Correct password verifies
    assert verify_secret(raw_password, hashed)

    # Wrong password fails
    assert not verify_secret("WrongPassword@123", hashed)

def test_security_answer_normalization():
    answer = "  Oxford University  "
    hashed = hash_security_answer(answer)

    # Check varying case and surrounding whitespace
    assert verify_security_answer("oxford university", hashed)
    assert verify_security_answer("OXFORD UNIVERSITY", hashed)
    assert verify_security_answer("   Oxford University   ", hashed)
    assert not verify_security_answer("Cambridge University", hashed)

def test_jwt_generation_and_verification():
    token = generate_jwt_token(user_id=42, username="test_user", email="test@example.com", expiry_hours=1)
    assert isinstance(token, str)

    is_valid, payload, msg = verify_jwt_token(token)
    assert is_valid
    assert payload is not None
    assert payload["sub"] == "42"
    assert payload["username"] == "test_user"
    assert payload["email"] == "test@example.com"
    assert msg == "Valid session."

def test_jwt_tampered_token():
    token = generate_jwt_token(user_id=1, username="alice", email="alice@example.com")
    tampered_token = token[:-5] + "abcde"

    is_valid, payload, msg = verify_jwt_token(tampered_token)
    assert not is_valid
    assert payload is None
    assert "Invalid session token" in msg

def test_jwt_expired_token():
    # Construct an expired token manually
    expired_payload = {
        "sub": "1",
        "username": "expired_user",
        "email": "exp@example.com",
        "iat": int(time.time()) - 7200,
        "exp": int(time.time()) - 3600
    }
    expired_token = jwt.encode(expired_payload, JWT_SECRET, algorithm=JWT_ALGORITHM)

    is_valid, payload, msg = verify_jwt_token(expired_token)
    assert not is_valid
    assert payload is None
    assert "Session has expired" in msg


In [ ]:
%%writefile tests/test_otp.py
"""
Unit tests for OTP generation, database hashing, attempt limits, expiry, and separation of purposes.
"""

import time
import os
import tempfile
import pytest
from src.database import init_db, get_connection
from src.otp_service import (
    generate_numeric_otp,
    create_and_store_otp,
    verify_stored_otp
)

@pytest.fixture
def temp_db():
    fd, path = tempfile.mkstemp(suffix=".db")
    os.close(fd)
    init_db(path)
    yield path
    if os.path.exists(path):
        os.remove(path)

def test_numeric_otp_format():
    otp = generate_numeric_otp(6)
    assert len(otp) == 6
    assert otp.isdigit()

def test_otp_creation_and_hashing(temp_db):
    email = "test@example.com"
    success, otp_code, msg = create_and_store_otp(email, "SIGNUP", temp_db)
    assert success
    assert len(otp_code) == 6

    # Verify that plaintext OTP is NOT in database
    conn = get_connection(temp_db)
    cursor = conn.cursor()
    cursor.execute("SELECT otp_hash FROM otp_codes WHERE identifier = ?", (email,))
    row = cursor.fetchone()
    conn.close()

    assert row is not None
    assert row["otp_hash"].startswith("$2b$")
    assert otp_code not in row["otp_hash"]

def test_otp_successful_verification(temp_db):
    email = "student@infosys.com"
    _, otp_code, _ = create_and_store_otp(email, "SIGNUP", temp_db)

    is_valid, msg = verify_stored_otp(email, "SIGNUP", otp_code, temp_db)
    assert is_valid
    assert "verified successfully" in msg.lower()

    # Once verified, it cannot be reused (one-time verification)
    is_valid_again, msg_again = verify_stored_otp(email, "SIGNUP", otp_code, temp_db)
    assert not is_valid_again
    assert "No active OTP request" in msg_again

def test_otp_separate_purposes(temp_db):
    # Ensure signup OTP and password-recovery OTP are separate
    email = "dual@example.com"
    _, signup_otp, _ = create_and_store_otp(email, "SIGNUP", temp_db)
    _, recovery_otp, _ = create_and_store_otp(email, "FORGOT_PASSWORD", temp_db)

    # Trying to verify signup OTP with FORGOT_PASSWORD purpose must fail
    is_valid, _ = verify_stored_otp(email, "FORGOT_PASSWORD", signup_otp, temp_db)
    assert not is_valid

    # Recovery OTP must verify with FORGOT_PASSWORD purpose
    is_valid, _ = verify_stored_otp(email, "FORGOT_PASSWORD", recovery_otp, temp_db)
    assert is_valid

def test_otp_attempt_limits(temp_db):
    email = "limit@example.com"
    _, real_otp, _ = create_and_store_otp(email, "SIGNUP", temp_db)

    # 1st wrong attempt
    valid, msg = verify_stored_otp(email, "SIGNUP", "000000", temp_db)
    assert not valid
    assert "2 attempt(s) remaining" in msg

    # 2nd wrong attempt
    valid, msg = verify_stored_otp(email, "SIGNUP", "111111", temp_db)
    assert not valid
    assert "1 attempt(s) remaining" in msg

    # 3rd wrong attempt: should invalidate
    valid, msg = verify_stored_otp(email, "SIGNUP", "222222", temp_db)
    assert not valid
    assert "Maximum attempts" in msg

    # Now even the correct OTP should be rejected because account is blocked for 15 minutes
    valid, msg = verify_stored_otp(email, "SIGNUP", real_otp, temp_db)
    assert not valid
    assert ("blocked" in msg.lower() or "locked" in msg.lower())

    # Requesting a new OTP during this 15-minute lockout period must also be rejected
    can_request, _, req_msg = create_and_store_otp(email, "SIGNUP", temp_db)
    assert not can_request
    assert ("blocked" in req_msg.lower() or "locked" in req_msg.lower())

def test_otp_expired(temp_db):
    email = "expired@example.com"
    _, otp_code, _ = create_and_store_otp(email, "SIGNUP", temp_db)

    # Artificially expire the OTP in database
    conn = get_connection(temp_db)
    cursor = conn.cursor()
    cursor.execute("UPDATE otp_codes SET expires_at = ? WHERE identifier = ?", (time.time() - 100, email))
    conn.commit()
    conn.close()

    valid, msg = verify_stored_otp(email, "SIGNUP", otp_code, temp_db)
    assert not valid
    assert "expired" in msg.lower()


In [ ]:
%%writefile tests/test_auth.py
"""
Integration tests for complete authentication flows:
1. Signup staging & OTP-gated permanent account creation.
2. Login with username or email, brute-force lockout, and JWT token generation.
3. Forgot password Route A (Security question) & Route B (Email OTP).
4. Admin authentication and credential masking in user catalog.
"""

import os
import tempfile
import pytest
from src.database import init_db, get_connection
from src.config import SECURITY_QUESTIONS, ADMIN_USERNAME, ADMIN_PASSWORD
from src.security import verify_jwt_token
from src.auth_service import (
    register_user,
    stage_signup,
    complete_signup,
    login_user,
    get_security_question_for_user,
    reset_password_with_security_question,
    initiate_forgot_password_otp,
    reset_password_with_otp,
    authenticate_admin,
    get_all_registered_users
)

@pytest.fixture
def auth_db():
    fd, path = tempfile.mkstemp(suffix=".db")
    os.close(fd)
    init_db(path)
    yield path
    if os.path.exists(path):
        os.remove(path)

def test_signup_otp_gated_flow(auth_db):
    username = "alice_wonder"
    email = "alice@wonderland.org"
    password = "SecurePassword@123"
    question = SECURITY_QUESTIONS[0]
    answer = "Greenwood High"

    # 1. Stage signup
    ok, msg, dev_otp = stage_signup(
        username=username,
        email=email,
        password=password,
        confirm_password=password,
        security_question=question,
        security_answer=answer,
        db_path=auth_db
    )
    assert ok, f"Staging signup failed: {msg}"
    assert dev_otp is not None

    conn = get_connection(auth_db)
    cursor = conn.cursor()

    # 2. Verify account is NOT in permanent users table yet
    cursor.execute("SELECT * FROM users WHERE email = ?", (email,))
    assert cursor.fetchone() is None, "Permanent user account should NOT exist before OTP verification!"

    # 3. Verify record IS staged in pending_signups
    cursor.execute("SELECT * FROM pending_signups WHERE email = ?", (email,))
    staged = cursor.fetchone()
    assert staged is not None
    assert staged["username"] == username
    conn.close()

    # 4. Attempt login before OTP verification -> Must fail
    login_ok, _, _, _ = login_user(username, password, db_path=auth_db)
    assert not login_ok

    # 5. Complete signup with wrong OTP -> Fails
    comp_ok, comp_msg = complete_signup(email, "999999", db_path=auth_db)
    assert not comp_ok
    assert "Incorrect OTP" in comp_msg

    # 6. Complete signup with valid OTP -> Succeeds
    comp_ok, comp_msg = complete_signup(email, dev_otp, db_path=auth_db)
    assert comp_ok
    assert "successfully verified" in comp_msg.lower()

    # 7. Check permanent users table now contains user
    conn = get_connection(auth_db)
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM users WHERE email = ?", (email,))
    user = cursor.fetchone()
    assert user is not None
    assert user["username"] == username
    # Password hash must be bcrypt
    assert user["password_hash"].startswith("$2b$")
    assert user["security_answer_hash"].startswith("$2b$")

    # Pending signups record must be cleaned
    cursor.execute("SELECT * FROM pending_signups WHERE email = ?", (email,))
    assert cursor.fetchone() is None
    conn.close()

def test_login_with_username_and_email(auth_db):
    username = "bob_builder"
    email = "bob@builder.com"
    password = "BuilderPass@2026"
    question = SECURITY_QUESTIONS[1]
    answer = "Smith"

    # Stage and verify
    _, _, dev_otp = stage_signup(username, email, password, password, question, answer, db_path=auth_db)
    complete_signup(email, dev_otp, db_path=auth_db)

    # 1. Login with username
    ok, token, msg, user_info = login_user(username, password, db_path=auth_db)
    assert ok
    assert token is not None
    assert user_info["username"] == username
    valid_jwt, payload, _ = verify_jwt_token(token)
    assert valid_jwt
    assert payload["username"] == username

    # 2. Login with email
    ok, token2, msg, user_info2 = login_user(email, password, db_path=auth_db)
    assert ok
    assert token2 is not None
    assert user_info2["email"] == email

    # 3. Wrong password -> generic error
    ok, token3, err_msg, _ = login_user(username, "WrongPassword@123", db_path=auth_db)
    assert not ok
    assert token3 is None
    assert err_msg == "Invalid username/email or password."

def test_login_brute_force_lockout(auth_db):
    username = "target_user"
    email = "target@example.com"
    password = "TargetPassword@123"

    _, _, dev_otp = stage_signup(username, email, password, password, SECURITY_QUESTIONS[0], "Answer", db_path=auth_db)
    complete_signup(email, dev_otp, db_path=auth_db)

    # 3 consecutive failed logins
    for i in range(3):
        ok, _, msg, _ = login_user(username, f"WrongPassword@{i}", db_path=auth_db)
        assert not ok

    # 4th login attempt should report account locked
    ok, _, msg, _ = login_user(username, password, db_path=auth_db)
    assert not ok
    assert "locked" in msg.lower()

def test_forgot_password_route_a_security_question(auth_db):
    username = "charlie"
    email = "charlie@domain.com"
    password = "InitialPassword@123"
    question = SECURITY_QUESTIONS[2]
    answer = "Goldie"

    _, _, dev_otp = stage_signup(username, email, password, password, question, answer, db_path=auth_db)
    complete_signup(email, dev_otp, db_path=auth_db)

    # Retrieve question
    ok, retrieved_q, _ = get_security_question_for_user(username, db_path=auth_db)
    assert ok
    assert retrieved_q == question

    # Wrong answer -> fails
    new_password = "UpdatedPassword@2026"
    ok, msg = reset_password_with_security_question(username, "WrongAnswer", new_password, new_password, db_path=auth_db)
    assert not ok
    assert "incorrect" in msg.lower()

    # Correct answer -> succeeds
    ok, msg = reset_password_with_security_question(username, "goldie", new_password, new_password, db_path=auth_db)
    assert ok

    # Verify login with new password
    login_ok, _, _, _ = login_user(username, new_password, db_path=auth_db)
    assert login_ok

def test_forgot_password_route_b_email_otp(auth_db):
    username = "diana"
    email = "diana@domain.com"
    password = "DianaPassword@123"

    _, _, dev_otp = stage_signup(username, email, password, password, SECURITY_QUESTIONS[0], "Ans", db_path=auth_db)
    complete_signup(email, dev_otp, db_path=auth_db)

    # Request recovery OTP
    ok, msg, rec_otp = initiate_forgot_password_otp(email, db_path=auth_db)
    assert ok
    assert rec_otp is not None

    # Reset password with OTP
    new_pass = "DianaNewPassword@2026"
    ok, msg = reset_password_with_otp(email, rec_otp, new_pass, new_pass, db_path=auth_db)
    assert ok

    # Verify login with updated password
    login_ok, _, _, _ = login_user(email, new_pass, db_path=auth_db)
    assert login_ok

def test_admin_dashboard_security_and_masking(auth_db):
    # Verify admin credentials check
    assert authenticate_admin(ADMIN_USERNAME, ADMIN_PASSWORD)
    assert not authenticate_admin("admin", "WrongAdminPass")

    # Register sample user
    username = "eva_green"
    email = "eva@green.org"
    _, _, dev_otp = stage_signup(username, email, "EvaPass@2026", "EvaPass@2026", SECURITY_QUESTIONS[0], "Ans", db_path=auth_db)
    complete_signup(email, dev_otp, db_path=auth_db)

    users_list = get_all_registered_users(db_path=auth_db)
    assert len(users_list) >= 1

    for u in users_list:
        assert "Username" in u
        assert "Email Address" in u
        # SECURITY REQUIREMENT: Never expose password hashes or security answer hashes
        assert "password" not in u
        assert "password_hash" not in u
        assert "security_answer" not in u
        assert "security_answer_hash" not in u

def test_direct_user_registration(auth_db):
    username = "direct_user"
    email = "direct@example.com"
    password = "DirectPass@2026"
    question = SECURITY_QUESTIONS[0]
    answer = "Central School"

    # Direct registration without OTP
    ok, msg = register_user(
        username=username,
        email=email,
        password=password,
        confirm_password=password,
        security_question=question,
        security_answer=answer,
        db_path=auth_db
    )
    assert ok, f"Registration failed: {msg}"
    assert "successfully registered" in msg.lower()

    # User immediately exists in users table
    conn = get_connection(auth_db)
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM users WHERE username = ?", (username,))
    row = cursor.fetchone()
    conn.close()
    assert row is not None
    assert row["email"] == email

    # User can log in immediately
    login_ok, token, _, user_info = login_user(username, password, db_path=auth_db)
    assert login_ok
    assert token is not None

def test_security_question_recovery_with_username_and_email(auth_db):
    username = "dual_rec_user"
    email = "dual_rec@example.com"
    password = "OriginalPassword@123"
    question = SECURITY_QUESTIONS[1]
    answer = "Pet Rover"

    # Register
    ok, _ = register_user(username, email, password, password, question, answer, db_path=auth_db)
    assert ok

    # Retrieve security question using USERNAME
    ok_user, q_user, _ = get_security_question_for_user(username, db_path=auth_db)
    assert ok_user
    assert q_user == question

    # Retrieve security question using EMAIL
    ok_email, q_email, _ = get_security_question_for_user(email, db_path=auth_db)
    assert ok_email
    assert q_email == question

    # Reset password using EMAIL identifier
    new_pass = "ResetViaEmail@2026"
    reset_ok, reset_msg = reset_password_with_security_question(
        email, "pet rover", new_pass, new_pass, db_path=auth_db
    )
    assert reset_ok
    assert "successfully reset" in reset_msg.lower()

    # Can log in with new password
    login_ok, _, _, _ = login_user(username, new_pass, db_path=auth_db)
    assert login_ok

def test_otp_lockout_across_operations(auth_db):
    from src.otp_service import verify_stored_otp

    username = "locked_user"
    email = "locked@example.com"
    password = "LockedPass@123"

    ok, _ = register_user(username, email, password, password, SECURITY_QUESTIONS[0], "Ans", db_path=auth_db)
    assert ok

    # Initiate OTP
    ok_otp, _, _ = initiate_forgot_password_otp(email, db_path=auth_db)
    assert ok_otp

    # 3 consecutive incorrect OTP entries
    for _ in range(2):
        v_ok, v_msg = verify_stored_otp(email, "FORGOT_PASSWORD", "000000", db_path=auth_db)
        assert not v_ok
        assert "remaining" in v_msg

    # 3rd incorrect attempt triggers 15-minute lockout
    v_ok, v_msg = verify_stored_otp(email, "FORGOT_PASSWORD", "000000", db_path=auth_db)
    assert not v_ok
    assert "blocked for 15 minutes" in v_msg.lower() or "maximum attempts" in v_msg.lower()

    # 1. No new OTP can be requested
    can_request, req_msg, _ = initiate_forgot_password_otp(email, db_path=auth_db)
    assert not can_request
    assert "blocked" in req_msg.lower() or "locked" in req_msg.lower()

    # 2. Login is blocked for this account during lockout
    login_ok, _, log_msg, _ = login_user(username, password, db_path=auth_db)
    assert not login_ok
    assert "locked" in log_msg.lower() or "blocked" in log_msg.lower()

    # 3. Security question recovery is blocked during lockout
    sq_ok, _, sq_msg = get_security_question_for_user(email, db_path=auth_db)
    assert not sq_ok
    assert "locked" in sq_msg.lower() or "blocked" in sq_msg.lower()



In [ ]:
%%writefile tests/conftest.py
"""
Test setup: config.py refuses to load without JWT_SECRET and ADMIN_PASSWORD.
Real secrets (Colab Secrets / .env) win; these throw-away values only fill gaps for the tests.
"""

import os

os.environ.setdefault("JWT_SECRET", "test-only-jwt-secret-not-for-real-use-0123456789")
os.environ.setdefault("ADMIN_PASSWORD", "TestOnly@Admin1")


In [ ]:
%%writefile tests/test_roles.py
"""
Unit tests for the role and outlet columns, the seeded Admin account and role-carrying JWTs.
"""

import os
import tempfile
import pytest
from src.database import init_db, get_connection
from src.config import SECURITY_QUESTIONS, ADMIN_USERNAME, ADMIN_PASSWORD, ADMIN_ROLE, DEFAULT_ROLE
from src.security import verify_jwt_token
from src.auth_service import register_user, login_user, get_all_registered_users

@pytest.fixture
def role_db():
    fd, path = tempfile.mkstemp(suffix=".db")
    os.close(fd)
    init_db(path)
    yield path
    os.remove(path)

def test_admin_is_seeded_with_admin_role(role_db):
    conn = get_connection(role_db)
    row = conn.execute("SELECT role, password_hash FROM users WHERE username = ?", (ADMIN_USERNAME,)).fetchone()
    conn.close()
    assert row["role"] == ADMIN_ROLE
    assert row["password_hash"] != ADMIN_PASSWORD  # stored hashed, never plain

def test_init_db_does_not_duplicate_admin(role_db):
    init_db(role_db)
    conn = get_connection(role_db)
    count = conn.execute("SELECT COUNT(*) AS c FROM users WHERE username = ?", (ADMIN_USERNAME,)).fetchone()["c"]
    conn.close()
    assert count == 1

def test_new_user_gets_default_role_and_token_carries_it(role_db):
    ok, _ = register_user("role_user", "role@user.org", "RolePass@2026", "RolePass@2026",
                          SECURITY_QUESTIONS[0], "Ans", db_path=role_db)
    assert ok
    ok, token, _, info = login_user("role_user", "RolePass@2026", db_path=role_db)
    assert ok
    assert info["role"] == DEFAULT_ROLE
    valid, payload, _ = verify_jwt_token(token)
    assert valid and payload["role"] == DEFAULT_ROLE

def test_user_directory_lists_role_without_hashes(role_db):
    users = get_all_registered_users(db_path=role_db)
    assert all("Role" in u and "Outlet" in u for u in users)
    assert all("password_hash" not in u for u in users)

def test_old_database_is_upgraded(tmp_path):
    import sqlite3
    path = str(tmp_path / "old.db")
    conn = sqlite3.connect(path)
    conn.execute("""CREATE TABLE users (id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE NOT NULL,
        email TEXT UNIQUE NOT NULL, password_hash TEXT NOT NULL, security_question TEXT NOT NULL,
        security_answer_hash TEXT NOT NULL, created_at TEXT NOT NULL, updated_at TEXT NOT NULL)""")
    conn.execute("INSERT INTO users (username,email,password_hash,security_question,security_answer_hash,created_at,updated_at)"
                 " VALUES ('old','old@user.org','x','q','y','t','t')")
    conn.commit()
    conn.close()
    init_db(path)
    conn = get_connection(path)
    assert conn.execute("SELECT role FROM users WHERE username='old'").fetchone()["role"] == DEFAULT_ROLE
    conn.close()


In [ ]:
# Step 4: Run Complete Automated Test Suite in Colab
!python -m pytest tests -v

### Launch Streamlit Portal with ngrok Tunnel
Run the cell below to launch the Streamlit application and establish a secure public ngrok tunnel.
Open the printed Public URL in your browser.

In [ ]:
# Step 5: Launch Streamlit & ngrok Tunnel
import subprocess
import time
import os
from pathlib import Path
from pyngrok import ngrok

# Bridge Colab Secrets (userdata) into environment variables and .env
# because Streamlit runs as a separate subprocess where google.colab.userdata is not accessible!
secrets_to_export = [
    "EMAIL_ADDRESS", "EMAIL_PASSWORD", "JWT_SECRET",
    "ADMIN_USERNAME", "ADMIN_PASSWORD", "NGROK_AUTHTOKEN",
    "\DDRESS", "ADDRESS", "EMAIL"
]

exported_env = os.environ.copy()
env_file_lines = []

try:
    from google.colab import userdata
    for s in secrets_to_export:
        try:
            val = userdata.get(s)
            if val is not None and str(val).strip():
                clean_val = str(val).strip()
                canonical_key = "EMAIL_ADDRESS" if s in ["\DDRESS", "DDRESS", "ADDRESS", "EMAIL"] else s
                exported_env[canonical_key] = clean_val
                os.environ[canonical_key] = clean_val
                env_file_lines.append(f"{canonical_key}={clean_val}")
                print(f"[OK] Exported secret '{canonical_key}' to Streamlit environment.")
        except Exception:
            pass
except Exception:
    pass

# Write .env files so config.py loads them inside the Streamlit subprocess
with open(".env", "w") as f:
    f.write("\n".join(env_file_lines) + "\n")
with open("src/.env", "w") as f:
    f.write("\n".join(env_file_lines) + "\n")

NGROK_TOKEN = exported_env.get('NGROK_AUTHTOKEN', '')
if NGROK_TOKEN:
    try:
        ngrok.set_auth_token(NGROK_TOKEN)
    except Exception:
        pass  # Token already set in ngrok config, safe to continue
else:
    print("[WARNING] NGROK_AUTHTOKEN is not configured. ngrok may fail without an authtoken.")

ngrok.kill()
!pkill -f streamlit

# Ensure user-level config also disables CORS & XSRF for ngrok WebSockets
user_streamlit_dir = os.path.expanduser("~/.streamlit")
os.makedirs(user_streamlit_dir, exist_ok=True)
with open(os.path.join(user_streamlit_dir, "config.toml"), "w") as f:
    f.write('''[server]
headless = true
enableCORS = false
enableXsrfProtection = false
enableWebsocketCompression = false
[browser]
gatherUsageStats = false
''')

process = subprocess.Popen(
    [
        "streamlit", "run", "src/app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.enableWebsocketCompression", "false",
        "--browser.gatherUsageStats", "false"
    ],
    env=exported_env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(3)

tunnel = ngrok.connect(8501)
print("=" * 65)
print(f"Infosys Portal Live URL: {tunnel.public_url}")
print("=" * 65)
print("Application is running. Press [Ctrl + C] or click Colab Interrupt to stop.")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nShutting down...")
    ngrok.kill()
    process.terminate()
    !pkill -f streamlit
    print("[OK] Streamlit server and ngrok tunnel stopped gracefully.")
